# Optimum Investments · Optimización del riesgo patrimonial (OP01)

Notebook **autocontenido** de la entrega: recibe `input.json` (contrato v1.0, Anexo 1), muestra los datos,
optimiza la cartera de activos y pasivos y escribe `output.json`.

$$\max_{x,y}\; E[\Delta PN] - \lambda\,\mathrm{CVaR}_\alpha(-\Delta PN) - TC$$

sujeto a presupuestos, límites de composición y restricciones comunes. Se resuelve como LP de
Rockafellar–Uryasev sobre escenarios a un año (bootstrap por bloques + estrés) revalorizados flujo a flujo.

**Cómo usarlo**
1. Deje `input.json` en la misma carpeta que este notebook (o escriba su ruta en `INPUT_PATH`; si no existe,
   el notebook la pide).
2. *Run All*. Tarda 2–4 minutos (pruebas incluidas); con `RUN_SENSITIVITY = True`, unos 6–8 minutos más.
3. El resultado queda en `OUTPUT_PATH` (por defecto `output.json`, junto al notebook).

**Requisitos:** Python ≥ 3.11 con `numpy pandas scipy cvxpy highspy clarabel matplotlib`
(`pip install numpy pandas scipy cvxpy highspy clarabel matplotlib`).

Ningún instrumento, límite, λ, α, costo o presupuesto está fijo en el código: todo sale de `input.json`.
Convenciones: montos en S/ millones, tasas en decimales, tiempo ACT/365; PN = V(activos) − V(pasivos) y
pérdida = −ΔPN.

In [ ]:
# Parámetros de ejecución
INPUT_PATH = "input.json"  # ruta al input; si no existe, se pide por teclado
OUTPUT_PATH = "output.json"  # dónde se escribe el resultado
RUN_SENSITIVITY = False  # True: corre las sensibilidades S1–S9 (≈ 6–8 min) y escribe sensitivity.json
SAVE_FIGURES = False  # True: guarda cada figura como PNG en la carpeta figuras/

## 1. Entorno

In [ ]:
import importlib
import sys

_missing = [m for m in ["numpy", "pandas", "scipy", "cvxpy", "highspy", "clarabel", "matplotlib"]
            if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(f"Faltan paquetes: {_missing}. Instalar con: pip install {' '.join(_missing)}")

import cvxpy as cp

_solvers = {"HIGHS", "CLARABEL"} - set(cp.installed_solvers())
if _solvers:
    raise ImportError(f"cvxpy no encuentra los solvers {_solvers}: pip install highspy clarabel")
print(f"Python {sys.version.split()[0]} · cvxpy {cp.__version__} · solvers OK")

## 2. Código del modelo

Cada celda escribe un módulo del paquete `optimum` en una carpeta temporal (copia exacta del código fuente del proyecto). Las capas son: (i) contrato JSON, (ii) valorización por flujos, (iii) riesgo y escenarios, (iv) optimizador y (v) sensibilidades.

In [ ]:
# Paquete temporal `optimum` con el código del modelo (las celdas %%writefile siguientes lo llenan)
import tempfile
from pathlib import Path

LIB = tempfile.mkdtemp(prefix="optimum_lib_")
for sub in ["optimum/io", "optimum/reporting"]:
    Path(LIB, sub).mkdir(parents=True, exist_ok=True)
for name in ['__init__.py', 'io/__init__.py', 'reporting/__init__.py']:
    Path(LIB, "optimum", name).touch()
sys.path.insert(0, LIB)
print("Código del modelo en", LIB)

### Factores de riesgo (extracto de `cleaning.py`)

In [ ]:
%%writefile {LIB}/optimum/cleaning.py
"""Limpieza: data/interim (hojas crudas) -> data/processed (tablas tipadas y validadas). (extracto para el notebook de entrega)."""

from __future__ import annotations
import numpy as np
import pandas as pd

RATE_NODES = ["ON", "1Y", "3Y", "5Y", "10Y", "20Y"]

SEGMENTS = ["S1", "S2", "S3", "S4", "S5"]

CURRENCIES = ["PEN", "USD"]

SPOT_FACTORS = [f"{c}_SPOT_{n}" for c in CURRENCIES for n in RATE_NODES]

FWD_FACTORS = [f"{c}_FWD_{s}" for c in CURRENCIES for s in SEGMENTS]

FACTORS = SPOT_FACTORS + FWD_FACTORS + ["FX_PENUSD", "EQUITY_USD"]  # 24 factores

RATE_FACTORS = SPOT_FACTORS + FWD_FACTORS  # cambian por diferencia absoluta

LOG_FACTORS = ["FX_PENUSD", "EQUITY_USD"]  # cambian por log-retorno


def compute_factor_changes(history: pd.DataFrame) -> pd.DataFrame:
    """Cambios mensuales: Δ absoluta en tasas, log-retorno en FX y equity."""
    out = history[RATE_FACTORS].diff()
    out[LOG_FACTORS] = np.log(history[LOG_FACTORS]).diff()
    return out[FACTORS].dropna()

### Curvas cero-cupón

Curvas spot cero-cupón: convenciones del caso (sección "Información histórica").

- Tasas spot efectivas anuales; tiempo en años ACT/365 desde la fecha de valorización.
- Interpolación LINEAL de la tasa cero entre nodos; extrapolación plana fuera de [ON, 20Y].
- Nodo ON = plazo 0 años.
- Factor de descuento con spread de calibración s: DF(t) = (1 + z(t) + s) ** (-t).

In [ ]:
%%writefile {LIB}/optimum/curves.py
"""Curvas spot cero-cupón: convenciones del caso (sección "Información histórica").

- Tasas spot efectivas anuales; tiempo en años ACT/365 desde la fecha de valorización.
- Interpolación LINEAL de la tasa cero entre nodos; extrapolación plana fuera de [ON, 20Y].
- Nodo ON = plazo 0 años.
- Factor de descuento con spread de calibración s: DF(t) = (1 + z(t) + s) ** (-t).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

NODE_YEARS = {"ON": 0.0, "1Y": 1.0, "3Y": 3.0, "5Y": 5.0, "10Y": 10.0, "20Y": 20.0}
SEGMENT_BOUNDS = {
    "S1": ("ON", "1Y"),
    "S2": ("1Y", "3Y"),
    "S3": ("3Y", "5Y"),
    "S4": ("5Y", "10Y"),
    "S5": ("10Y", "20Y"),
}


def year_fraction(start, end) -> np.ndarray:
    """ACT/365 entre fechas (escalares, arrays, DatetimeIndex o pd.Series)."""
    delta = pd.to_datetime(end) - pd.Timestamp(start)
    days = np.asarray(pd.to_timedelta(delta) / pd.Timedelta(days=1), dtype=float)
    return days / 365.0


@dataclass(frozen=True)
class ZeroCurve:
    currency: str
    tenors: np.ndarray  # años, creciente
    rates: np.ndarray  # tasas efectivas anuales, decimales

    @classmethod
    def from_market_row(cls, row: pd.Series, currency: str) -> ZeroCurve:
        """Construye la curva desde una fila de market_history (p. ej. la del 2025-12-31)."""
        tenors = np.array(list(NODE_YEARS.values()))
        rates = np.array([row[f"{currency}_SPOT_{n}"] for n in NODE_YEARS], dtype=float)
        return cls(currency, tenors, rates)

    def zero(self, t) -> np.ndarray:
        return np.interp(np.asarray(t, dtype=float), self.tenors, self.rates)

    def discount(self, t, spread: float = 0.0) -> np.ndarray:
        t = np.asarray(t, dtype=float)
        return (1.0 + self.zero(t) + spread) ** (-t)

    def shifted(self, shifts) -> ZeroCurve:
        """Nueva curva con shocks por nodo (array de 6) o paralelo (escalar)."""
        return ZeroCurve(self.currency, self.tenors, self.rates + np.asarray(shifts, dtype=float))


def implied_forward(curve: ZeroCurve, t1: float, t2: float) -> float:
    """Forward efectiva anual implícita entre t1 y t2 (útil para chequear las FWD del Excel)."""
    g1 = (1 + curve.zero(t1)) ** t1
    g2 = (1 + curve.zero(t2)) ** t2
    return float((g2 / g1) ** (1.0 / (t2 - t1)) - 1.0)

### (i) Contrato JSON (extracto de `io/json_contract.py`)

In [ ]:
%%writefile {LIB}/optimum/io/json_contract.py
"""Contrato JSON v1.0 (Anexo 1 del caso): construcción, validación y escritura. (extracto para el notebook de entrega)."""

from __future__ import annotations
import math
from typing import Any
import numpy as np
import pandas as pd
from optimum.cleaning import FACTORS

SCHEMA_VERSION = "1.0"

REQUIRED_INPUT_KEYS = [
    "schemaVersion",
    "caseId",
    "valuationDate",
    "horizonDate",
    "baseCurrency",
    "curveNodes",
    "curveSegments",
    "marketHistory",
    "instruments",
    "cashFlows",
    "calibrationSpreads",
    "factorMapping",
    "constraints",
    "transactionCosts",
    "fundingAlternatives",
    "caseParameters",
]

REQUIRED_OUTPUT_KEYS = [
    "schemaVersion",
    "caseId",
    "status",
    "objectiveValue",
    "valuation",
    "metrics",
    "riskModel",
    "positions",
    "weights",
    "constraintChecks",
]


def _clean(v: Any) -> Any:
    """Convierte tipos numpy/pandas a JSON puro; NaN -> None."""
    if isinstance(v, dict):
        return {k: _clean(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_clean(x) for x in v]
    if isinstance(v, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(v).strftime("%Y-%m-%d")
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return None if math.isnan(v) else float(v)
    if v is pd.NA or v is pd.NaT:
        return None
    return v


def validate_input(doc: dict) -> list[str]:
    """Devuelve la lista de errores (vacía si el input es válido). Anexo 2, prompt 2."""
    errors: list[str] = []
    missing = [k for k in REQUIRED_INPUT_KEYS if k not in doc]
    if missing:
        return [f"Faltan claves: {missing}"]
    if doc["schemaVersion"] != SCHEMA_VERSION:
        errors.append(f"schemaVersion {doc['schemaVersion']} != {SCHEMA_VERSION}")
    if doc["caseId"] not in list("ABCDE"):
        errors.append(f"caseId inválido: {doc['caseId']}")

    node_ids = {n["id"] for n in doc["curveNodes"]}
    for seg in doc["curveSegments"]:
        if seg["fromNode"] not in node_ids or seg["toNode"] not in node_ids:
            errors.append(f"Segmento {seg['id']} referencia nodos inexistentes")
    seg_ids = {s["id"] for s in doc["curveSegments"]}

    dates = [m["date"] for m in doc["marketHistory"]]
    if len(dates) != len(set(dates)):
        errors.append("Observaciones históricas duplicadas en marketHistory")
    if dates != sorted(dates):
        errors.append("marketHistory no está ordenado por fecha")
    for m in doc["marketHistory"]:
        for ccy, segs in m["forwardSwap"].items():
            if set(segs) - seg_ids:
                errors.append(f"{m['date']}: segmentos inexistentes {set(segs) - seg_ids} ({ccy})")

    factor_ids = set(doc["factorMapping"]["factorIds"])
    if not factor_ids <= set(FACTORS):
        errors.append(f"factorIds desconocidos: {factor_ids - set(FACTORS)}")
    inst_ids = {i["id"] for i in doc["instruments"]}
    for inst in doc["instruments"] + doc["fundingAlternatives"]:
        ref = inst.get("reference_factor")
        if ref and ref not in factor_ids:
            errors.append(f"{inst['id']}: cupón flotante referencia factor no definido {ref}")
    for cf in doc["cashFlows"]:
        if cf["instrument_id"] not in inst_ids:
            errors.append(f"Flujo de instrumento inexistente {cf['instrument_id']}")
            break
    if set(doc["factorMapping"]["rows"]) != inst_ids:
        errors.append("factorMapping.rows no cubre exactamente los instrumentos")
    return errors


def validate_output(doc: dict) -> list[str]:
    missing = [k for k in REQUIRED_OUTPUT_KEYS if k not in doc]
    errors = [f"Faltan claves en output: {missing}"] if missing else []
    if doc.get("status") not in {"OPTIMAL", "OPTIMAL_INACCURATE", "INFEASIBLE", "UNBOUNDED", "ERROR"}:
        errors.append(f"status no reconocido: {doc.get('status')}")
    return errors

### (ii) Valorización por flujos

(ii) Valorización de instrumentos desde flujos y curvas — independiente del optimizador.

Convenciones (docs/formulacion.md §7.4; decisiones D-10, D-12, D-13):
- Descuento: DF(t) = (1 + z(t) + s)^(-t), t en años ACT/365 desde la fecha de valorización; s = spread de
  calibración (constante, recalibrado para que V0 = market_value_pen).
- Fijo: flujos contractuales de cashFlows.
- Flotante (D-13): cupón = nivel del factor de referencia (p. ej. PEN_FWD_S2) en la fecha de fijación + spread
  contractual; monto = tasa × nocional / frecuencia. Fijación = pago − reset_months (fin de mes si el pago lo es).
  Los cupones fijados en t0 o antes conservan la proyección base. Alternativa PERIOD_FORWARD (D-22, solo para la
  sensibilidad S9): forward implícita del período fijación → pago; se elige con `floatingCouponRule`.
- Forwards: siempre implícitas de la curva spot del estado (D-10), nunca shockeadas por separado.
- Trayectoria dentro del año (D-12): F(τ) = F0 + θΔ, θ = (τ − t0)/(tH − t0) acotado a [0, 1].
- Montos en moneda nativa hasta la conversión final a PEN con el FX del estado (soles por dólar).
- Nunca se usa duración/DV01: las sensibilidades salen de revalorizar.

In [ ]:
%%writefile {LIB}/optimum/valuation.py
"""(ii) Valorización de instrumentos desde flujos y curvas — independiente del optimizador.

Convenciones (docs/formulacion.md §7.4; decisiones D-10, D-12, D-13):
- Descuento: DF(t) = (1 + z(t) + s)^(-t), t en años ACT/365 desde la fecha de valorización; s = spread de
  calibración (constante, recalibrado para que V0 = market_value_pen).
- Fijo: flujos contractuales de cashFlows.
- Flotante (D-13): cupón = nivel del factor de referencia (p. ej. PEN_FWD_S2) en la fecha de fijación + spread
  contractual; monto = tasa × nocional / frecuencia. Fijación = pago − reset_months (fin de mes si el pago lo es).
  Los cupones fijados en t0 o antes conservan la proyección base. Alternativa PERIOD_FORWARD (D-22, solo para la
  sensibilidad S9): forward implícita del período fijación → pago; se elige con `floatingCouponRule`.
- Forwards: siempre implícitas de la curva spot del estado (D-10), nunca shockeadas por separado.
- Trayectoria dentro del año (D-12): F(τ) = F0 + θΔ, θ = (τ − t0)/(tH − t0) acotado a [0, 1].
- Montos en moneda nativa hasta la conversión final a PEN con el FX del estado (soles por dólar).
- Nunca se usa duración/DV01: las sensibilidades salen de revalorizar.
"""

from __future__ import annotations

import math
from collections.abc import Callable, Mapping
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import brentq

from optimum.curves import NODE_YEARS, SEGMENT_BOUNDS, ZeroCurve, implied_forward, year_fraction

PAYMENTS_PER_YEAR = {"ANNUAL": 1, "SEMIANNUAL": 2, "QUARTERLY": 4, "MONTHLY": 12}
COUPON_RULES = ("FLAT", "PERIOD_FORWARD")  # floatingCouponRule (D-13 base, D-22)
FX_FACTOR = "FX_PENUSD"
EQUITY_FACTOR = "EQUITY_USD"

MarketAt = Callable[[pd.Timestamp], "MarketState"]


# ------------------------------------------------------------------ estado de mercado
@dataclass(frozen=True)
class MarketState:
    """Estado de mercado: curvas spot por moneda, forwards implícitas por segmento, FX y equity."""

    curves: dict[str, ZeroCurve]  # {"PEN": ..., "USD": ...}; tasas efectivas anuales (decimales)
    forwards: dict[str, float]  # {"PEN_FWD_S1": ..., ...}; implícitas de `curves`
    fx_penusd: float  # soles por dólar
    equity_index: float  # nivel del índice (USD)

    @classmethod
    def from_curves(cls, curves: dict[str, ZeroCurve], fx_penusd: float, equity_index: float) -> MarketState:
        forwards = {
            f"{ccy}_FWD_{seg}": implied_forward(curve, NODE_YEARS[a], NODE_YEARS[b])
            for ccy, curve in curves.items()
            for seg, (a, b) in SEGMENT_BOUNDS.items()
        }
        return cls(curves, forwards, float(fx_penusd), float(equity_index))

    @classmethod
    def from_market_row(cls, row: Mapping) -> MarketState:
        """Desde una fila de `marketHistory` de input.json (spot por moneda y nodo, fx, equityIndex)."""
        tenors = np.array(list(NODE_YEARS.values()))
        curves = {
            ccy: ZeroCurve(ccy, tenors, np.array([nodes[n] for n in NODE_YEARS], dtype=float))
            for ccy, nodes in row["spot"].items()
        }
        return cls.from_curves(curves, row["fx"]["PENUSD"], row["equityIndex"])

    def shocked(self, delta: Mapping[str, float], theta: float = 1.0) -> MarketState:
        """Aplica θ·Δ: aditivo en nodos spot, multiplicativo e^(θΔ) en FX y equity (Δ = log-retorno).

        Los Δ de factores forward se ignoran: las forwards se recalculan desde la spot shockeada (D-10).
        """
        shifts = {ccy: np.zeros(len(NODE_YEARS)) for ccy in self.curves}
        fx, eq = self.fx_penusd, self.equity_index
        for name, d in delta.items():
            if name == FX_FACTOR:
                fx *= math.exp(theta * d)
            elif name == EQUITY_FACTOR:
                eq *= math.exp(theta * d)
            elif "_SPOT_" in name:
                ccy, node = name.split("_SPOT_")
                if ccy not in shifts or node not in NODE_YEARS:
                    raise ValueError(f"Factor spot desconocido: {name}")
                shifts[ccy][list(NODE_YEARS).index(node)] += theta * d
            elif "_FWD_" not in name:
                raise ValueError(f"Factor desconocido: {name}")
        curves = {ccy: c.shifted(shifts[ccy]) for ccy, c in self.curves.items()}
        return MarketState.from_curves(curves, fx, eq)

    def fx_to_pen(self, currency: str) -> float:
        if currency == "PEN":
            return 1.0
        if currency == "USD":
            return self.fx_penusd
        raise ValueError(f"Moneda no soportada: {currency}")


def market_path(state0: MarketState, delta: Mapping[str, float], t0, t_h) -> MarketAt:
    """Mercado en la fecha τ bajo la trayectoria lineal F0 + θ(τ)Δ, θ acotado a [0, 1] (D-12)."""
    t0, t_h = pd.Timestamp(t0), pd.Timestamp(t_h)
    span = float(year_fraction(t0, t_h))
    cache: dict[pd.Timestamp, MarketState] = {}

    def at(date) -> MarketState:
        date = pd.Timestamp(date)
        if date not in cache:
            theta = min(max(float(year_fraction(t0, date)) / span, 0.0), 1.0)
            cache[date] = state0.shocked(delta, theta)
        return cache[date]

    return at


# ------------------------------------------------------------------------ descuento
def present_value(flows: pd.DataFrame, curve: ZeroCurve, as_of, spread: float = 0.0) -> float:
    """Σ amount · (1 + z(t) + s)^(-t) sobre flujos con payment_date > as_of; t ACT/365 desde as_of.

    `flows`: columnas payment_date, amount (moneda nativa). Devuelve moneda nativa.
    """
    as_of = pd.Timestamp(as_of)
    dates = pd.to_datetime(flows["payment_date"])
    live = dates > as_of
    if not live.any():
        return 0.0
    t = year_fraction(as_of, dates[live])
    return float(np.sum(flows.loc[live, "amount"].to_numpy() * curve.discount(t, spread)))


# ------------------------------------------------------------------ cupones y flujos
def _fixing_date(payment: pd.Timestamp, months: int) -> pd.Timestamp:
    fix = payment - pd.DateOffset(months=months)
    return fix + pd.offsets.MonthEnd(0) if payment.is_month_end else fix


def coupon_rates(
    cf: pd.DataFrame, reset_months, t0, market_at: MarketAt, rule: str = "FLAT", as_of=None
) -> np.ndarray:
    """Tasa de cupón por período (decimal) de un instrumento, en el orden de `cf`.

    FIXED: tasa contractual. FLOAT: si la fijación es ≤ t0, la proyección base (ya fijada); si no, según `rule`:
    - FLAT (D-13): nivel del factor de referencia en `market_at(fijación)` + spread contractual.
    - PERIOD_FORWARD (D-22): con d = min(fijación, as_of), forward implícita efectiva anual de la curva de la moneda
      del factor en `market_at(d)` entre fijación − d y pago − d (años ACT/365) + spread. `as_of` es la fecha de
      valorización (t0 al calibrar, tH al horizonte); por defecto t0.
    """
    if rule not in COUPON_RULES:
        raise ValueError(f"floatingCouponRule desconocida: {rule} (válidas: {', '.join(COUPON_RULES)})")
    t0 = pd.Timestamp(t0)
    as_of = t0 if as_of is None else pd.Timestamp(as_of)
    rates = cf["projected_coupon_rate"].to_numpy(dtype=float).copy()
    is_float = (cf["coupon_type"] == "FLOAT").to_numpy()
    if not is_float.any():
        return rates
    if reset_months is None or pd.isna(reset_months):
        raise ValueError(f"Flotante sin reset_months: {cf['instrument_id'].iloc[0]}")
    for n in np.flatnonzero(is_float):
        row = cf.iloc[n]
        fix = _fixing_date(pd.Timestamp(row["payment_date"]), int(reset_months))
        if fix <= t0:
            continue
        ref = row["reference_factor"]
        if rule == "FLAT":
            state = market_at(fix)
            if ref not in state.forwards:
                raise ValueError(f"Factor de referencia desconocido {ref} en {row['instrument_id']}")
            rates[n] = state.forwards[ref] + float(row["spread"])
        else:
            d = min(fix, as_of)
            state = market_at(d)
            if ref not in state.forwards:
                raise ValueError(f"Factor de referencia desconocido {ref} en {row['instrument_id']}")
            curve = state.curves[ref.split("_")[0]]
            t1 = float(year_fraction(d, fix))
            t2 = float(year_fraction(d, pd.Timestamp(row["payment_date"])))
            rates[n] = implied_forward(curve, t1, t2) + float(row["spread"])
    return rates


def instrument_flows(
    inst: Mapping, cf: pd.DataFrame, t0, market_at: MarketAt, rule: str = "FLAT", as_of=None
) -> pd.DataFrame:
    """Flujos en moneda nativa (payment_date, amount): contractuales si es fijo, reproyectados si es flotante
    con la regla `rule` y fecha de información `as_of` (ver `coupon_rates`)."""
    amount = cf["total_cash_flow"].to_numpy(dtype=float).copy()
    is_float = (cf["coupon_type"] == "FLOAT").to_numpy()
    if is_float.any():
        freq = PAYMENTS_PER_YEAR.get(inst["frequency"])
        if freq is None:
            raise ValueError(f"Frecuencia no soportada en {inst['id']}: {inst['frequency']}")
        rates = coupon_rates(cf, inst["reset_months"], t0, market_at, rule, as_of)
        coupon = rates * cf["notional_native"].to_numpy(dtype=float) / freq
        amount[is_float] = coupon[is_float] + cf["principal_cash_flow"].to_numpy(dtype=float)[is_float]
    return pd.DataFrame({"payment_date": pd.to_datetime(cf["payment_date"]).to_numpy(), "amount": amount})


# ------------------------------------------------------------------- lectura del doc
def horizon_dates(doc: Mapping) -> tuple[pd.Timestamp, pd.Timestamp]:
    """(t0, tH): `valuationDate` y el horizonte (`caseParameters.horizonDate` o `horizonDate`)."""
    t_h = doc.get("caseParameters", {}).get("horizonDate") or doc["horizonDate"]
    return pd.Timestamp(doc["valuationDate"]), pd.Timestamp(t_h)


def coupon_rule(doc: Mapping) -> str:
    """`caseParameters.floatingCouponRule` (FLAT si falta, D-22)."""
    rule = doc.get("caseParameters", {}).get("floatingCouponRule") or "FLAT"
    if rule not in COUPON_RULES:
        raise ValueError(f"floatingCouponRule desconocida: {rule} (válidas: {', '.join(COUPON_RULES)})")
    return rule


def base_state(doc: Mapping) -> MarketState:
    """Mercado en la fecha de valorización (fila de marketHistory con esa fecha)."""
    t0 = pd.Timestamp(doc["valuationDate"])
    rows = [r for r in doc["marketHistory"] if pd.Timestamp(r["date"]) == t0]
    if not rows:
        raise ValueError(f"marketHistory no tiene la fecha de valorización {t0.date()}")
    return MarketState.from_market_row(rows[0])


def _cash_flows_by_instrument(doc: Mapping) -> dict[str, pd.DataFrame]:
    cf = pd.DataFrame(doc["cashFlows"])
    cf["payment_date"] = pd.to_datetime(cf["payment_date"])
    return {
        iid: g.sort_values("payment_date").reset_index(drop=True) for iid, g in cf.groupby("instrument_id")
    }


def _is_flow_instrument(inst: Mapping, cfs: Mapping) -> bool:
    return inst["instrument_type"] not in ("CASH", "EQUITY") and inst["id"] in cfs


# ---------------------------------------------------------------------- calibración
SPREAD_BRACKET = (-0.05, 0.50)  # intervalo de búsqueda numérica del spread (no es un parámetro del caso)


def check_positive_values(doc: Mapping) -> None:
    """`market_value_pen` es a la vez precio de calibración y posición inicial: debe ser > 0 en todo instrumento."""
    bad = [i["id"] for i in doc["instruments"] if not float(i["market_value_pen"] or 0.0) > 0]
    if bad:
        raise ValueError(
            f"market_value_pen debe ser > 0 (se usa para calibrar el precio y como base de r): {', '.join(bad)}"
        )


def calibrate_spreads(doc: Mapping) -> dict[str, float]:
    """Spread (decimal) por instrumento con flujos tal que V0 · FX0 = market_value_pen (tolerancia 1e-12)."""
    check_positive_values(doc)
    t0, _ = horizon_dates(doc)
    state0 = base_state(doc)
    rule = coupon_rule(doc)
    cfs = _cash_flows_by_instrument(doc)
    spreads = {}
    for inst in doc["instruments"]:
        if not _is_flow_instrument(inst, cfs):
            continue
        flows = instrument_flows(inst, cfs[inst["id"]], t0, lambda d: state0, rule, t0)
        curve, fx = state0.curves[inst["currency"]], state0.fx_to_pen(inst["currency"])
        target = float(inst["market_value_pen"])
        args = (flows, curve, t0, fx, target)
        lo, hi = SPREAD_BRACKET
        if _pricing_gap(lo, *args) * _pricing_gap(hi, *args) > 0:
            raise ValueError(
                f"No se puede calibrar {inst['id']}: el spread que da V0 = {target:,.2f} S/ mm está fuera de "
                f"[{lo:.0%}, {hi:.0%}]. Revisar market_value_pen y sus flujos."
            )
        spreads[inst["id"]] = brentq(_pricing_gap, lo, hi, args=args, xtol=1e-14)
    return spreads


def _pricing_gap(s: float, flows: pd.DataFrame, curve: ZeroCurve, t0, fx: float, target: float) -> float:
    return present_value(flows, curve, t0, s) * fx - target


# ---------------------------------------------------------------------- valorización
def value_at_t0(
    doc: Mapping, spreads: Mapping[str, float], delta: Mapping[str, float] | None = None
) -> pd.Series:
    """Valor en PEN (S/ mm) por instrumento en t0, con un shock instantáneo opcional Δ (sin trayectoria).

    Caja: nocional × FX. Equity: market_value_pen × (E/E0) × (FX/FX0 si es USD).
    """
    t0, _ = horizon_dates(doc)
    state0 = base_state(doc)
    state = state0.shocked(delta or {})
    rule = coupon_rule(doc)
    cfs = _cash_flows_by_instrument(doc)
    values = {}
    for inst in doc["instruments"]:
        iid, ccy = inst["id"], inst["currency"]
        fx_ratio = state.fx_to_pen(ccy) / state0.fx_to_pen(ccy)
        if inst["instrument_type"] == "CASH":
            values[iid] = float(inst["market_value_pen"]) * fx_ratio
        elif inst["instrument_type"] == "EQUITY":
            values[iid] = (
                float(inst["market_value_pen"]) * state.equity_index / state0.equity_index * fx_ratio
            )
        elif iid in cfs:
            flows = instrument_flows(inst, cfs[iid], t0, lambda d: state, rule, t0)
            values[iid] = present_value(flows, state.curves[ccy], t0, spreads[iid]) * state.fx_to_pen(ccy)
        else:
            raise ValueError(f"Instrumento sin flujos ni tipo valorizable: {iid}")
    return pd.Series(values, name="value_pen")


def horizon_value_and_flows(
    doc: Mapping, spreads: Mapping[str, float], delta: Mapping[str, float]
) -> pd.DataFrame:
    """Por instrumento, en PEN (S/ mm), bajo la trayectoria lineal del escenario Δ (D-12):

    - value_h: valor en tH de los flujos posteriores a tH, con el mercado F(tH) + spread calibrado.
    - flows: flujos cobrados/pagados en (t0, tH], cada uno convertido con FX(τ_pago); sin reinversión.
    Caja: devenga ON0 + ½ΔON (tasa simple ACT/365) y se reporta en value_h. Equity: sin dividendos.
    """
    t0, t_h = horizon_dates(doc)
    state0 = base_state(doc)
    path = market_path(state0, delta, t0, t_h)
    state_h = path(t_h)
    tau = float(year_fraction(t0, t_h))
    rule = coupon_rule(doc)
    cfs = _cash_flows_by_instrument(doc)
    rows = {}
    for inst in doc["instruments"]:
        iid, ccy = inst["id"], inst["currency"]
        if inst["instrument_type"] == "CASH":
            on0 = float(state0.curves[ccy].zero(0.0))
            accrued = 1.0 + (on0 + 0.5 * delta.get(f"{ccy}_SPOT_ON", 0.0)) * tau
            fx_ratio = state_h.fx_to_pen(ccy) / state0.fx_to_pen(ccy)
            rows[iid] = (float(inst["market_value_pen"]) * accrued * fx_ratio, 0.0)
        elif inst["instrument_type"] == "EQUITY":
            fx_ratio = state_h.fx_to_pen(ccy) / state0.fx_to_pen(ccy)
            growth = state_h.equity_index / state0.equity_index
            rows[iid] = (float(inst["market_value_pen"]) * growth * fx_ratio, 0.0)
        elif iid in cfs:
            flows = instrument_flows(inst, cfs[iid], t0, path, rule, t_h)
            in_year = (flows["payment_date"] > t0) & (flows["payment_date"] <= t_h)
            received = sum(
                a * path(d).fx_to_pen(ccy)
                for d, a in flows.loc[in_year, ["payment_date", "amount"]].itertuples(index=False)
            )
            value_h = present_value(flows, state_h.curves[ccy], t_h, spreads[iid]) * state_h.fx_to_pen(ccy)
            rows[iid] = (value_h, float(received))
        else:
            raise ValueError(f"Instrumento sin flujos ni tipo valorizable: {iid}")
    return pd.DataFrame.from_dict(rows, orient="index", columns=["value_h", "flows"])

### (iii) Factores, escenarios y CVaR

(iii) Factores, escenarios y medidas de riesgo — independiente del optimizador.

Convenciones (docs/formulacion.md §5, §7.1-7.4; decisiones D-06, D-09, D-11, D-14, D-15, D-16):
- 24 factores en el orden de `FACTORS`. Cambios mensuales: Δ absoluta en tasas, log-retorno en FX y equity.
- Bootstrap por bloques móviles: cada escenario suma `n_blocks` ventanas centradas de `block_length` meses
  consecutivos con inicio uniforme. Se añade la deriva anual μ escalada por (b·m)/12 (D-06, D-15). Con FX `IRP`,
  el log-shock FX se desplaza para que la media muestral de FX_T/FX_0 sea la forward de paridad (D-24).
- Estrés: shift paralelo por moneda; el override (un shift, D-16) reemplaza al paralelo en los nodos del tramo; en
  los de transición se interpola t·s(t) entre los nodos fijos vecinos, para que la forward cambie parejo (D-11 v2).
  Sin deriva.
- Forwards: sus Δ se ignoran al revalorizar; se recalculan desde la spot shockeada (D-10).
- r_{k,s} = (V_k(tH) + CF_k(t0, tH]) / V_k(t0) − 1, por revalorización completa (valuation.py).
- Pérdida = −ΔPN. VaR y CVaR con átomo fraccional y probabilidades por escenario.
- Todo parámetro sale del documento de input.json; nada se codifica por instrumento.

In [ ]:
%%writefile {LIB}/optimum/risk.py
"""(iii) Factores, escenarios y medidas de riesgo — independiente del optimizador.

Convenciones (docs/formulacion.md §5, §7.1-7.4; decisiones D-06, D-09, D-11, D-14, D-15, D-16):
- 24 factores en el orden de `FACTORS`. Cambios mensuales: Δ absoluta en tasas, log-retorno en FX y equity.
- Bootstrap por bloques móviles: cada escenario suma `n_blocks` ventanas centradas de `block_length` meses
  consecutivos con inicio uniforme. Se añade la deriva anual μ escalada por (b·m)/12 (D-06, D-15). Con FX `IRP`,
  el log-shock FX se desplaza para que la media muestral de FX_T/FX_0 sea la forward de paridad (D-24).
- Estrés: shift paralelo por moneda; el override (un shift, D-16) reemplaza al paralelo en los nodos del tramo; en
  los de transición se interpola t·s(t) entre los nodos fijos vecinos, para que la forward cambie parejo (D-11 v2).
  Sin deriva.
- Forwards: sus Δ se ignoran al revalorizar; se recalculan desde la spot shockeada (D-10).
- r_{k,s} = (V_k(tH) + CF_k(t0, tH]) / V_k(t0) − 1, por revalorización completa (valuation.py).
- Pérdida = −ΔPN. VaR y CVaR con átomo fraccional y probabilidades por escenario.
- Todo parámetro sale del documento de input.json; nada se codifica por instrumento.
"""

from __future__ import annotations

import math
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass

import numpy as np
import pandas as pd

from optimum.cleaning import FACTORS, LOG_FACTORS, RATE_FACTORS, SPOT_FACTORS, compute_factor_changes
from optimum.curves import year_fraction
from optimum.valuation import (
    base_state,
    calibrate_spreads,
    check_positive_values,
    horizon_dates,
    horizon_value_and_flows,
)

MONTHS_PER_YEAR = 12  # los cambios históricos son mensuales
PROB_TOL = 1e-9


# ------------------------------------------------------------------ historia y cambios
def history_from_doc(doc: Mapping) -> pd.DataFrame:
    """Niveles de los 24 factores (fechas × FACTORS) desde `marketHistory`, hasta `valuationDate` inclusive (sin
    información futura). Tasas en decimales."""
    rows = {}
    for r in doc["marketHistory"]:
        row = {}
        for ccy, nodes in r["spot"].items():
            row.update({f"{ccy}_SPOT_{n}": v for n, v in nodes.items()})
        for ccy, segs in r["forwardSwap"].items():
            row.update({f"{ccy}_FWD_{s}": v for s, v in segs.items()})
        row["FX_PENUSD"] = r["fx"]["PENUSD"]
        row["EQUITY_USD"] = r["equityIndex"]
        rows[pd.Timestamp(r["date"])] = row
    hist = pd.DataFrame.from_dict(rows, orient="index").sort_index()
    missing = set(FACTORS) - set(hist.columns)
    if missing:
        raise ValueError(f"marketHistory no trae los factores {sorted(missing)}")
    hist.index.name = "date"
    return hist.loc[: horizon_dates(doc)[0], FACTORS].astype(float)


def factor_changes(doc: Mapping) -> pd.DataFrame:
    """Cambios mensuales (Δ absoluta en tasas, log-retorno en FX y equity) desde `marketHistory`."""
    return compute_factor_changes(history_from_doc(doc))


# ------------------------------------------------------------------------- deriva
def drift(doc: Mapping, changes: pd.DataFrame) -> pd.Series:
    """Deriva anual μ por factor (decimales; log-retorno en FX y equity), según `caseParameters.drift` (D-06).

    - rates: ZERO → 0 | HISTORICAL → 12 × media mensual.
    - fx: ZERO | HISTORICAL | IRP → ln((1 + z_PEN(τ)) / (1 + z_USD(τ))), τ = plazo del horizonte, curvas de t0
      (FX = PEN por USD; tasa anual, de modo que μ·τ es la forward de paridad al horizonte).
    - equity: ZERO | HISTORICAL.
    """
    cfg = doc["caseParameters"]["drift"]
    historical = MONTHS_PER_YEAR * changes.mean()
    mu = pd.Series(0.0, index=FACTORS)

    def pick(kind: str, cls: str, factors: list[str], allowed: set[str]) -> None:
        if kind not in allowed:
            raise ValueError(f"Deriva '{kind}' no soportada para {cls} (opciones: {sorted(allowed)})")
        if kind == "HISTORICAL":
            mu[factors] = historical[factors]

    pick(cfg["rates"], "rates", RATE_FACTORS, {"ZERO", "HISTORICAL"})
    pick(cfg["fx"], "fx", ["FX_PENUSD"], {"ZERO", "HISTORICAL", "IRP"})
    pick(cfg["equity"], "equity", ["EQUITY_USD"], {"ZERO", "HISTORICAL"})
    if cfg["fx"] == "IRP":
        curves = base_state(doc).curves
        tau = float(year_fraction(*horizon_dates(doc)))
        mu["FX_PENUSD"] = math.log(
            (1 + float(curves["PEN"].zero(tau))) / (1 + float(curves["USD"].zero(tau)))
        )
    return mu


# ---------------------------------------------------------------------- bootstrap
def bootstrap_shocks(
    changes: pd.DataFrame, n: int, block_length: int, n_blocks: int, seed: int | None = None
) -> np.ndarray:
    """Shocks (n × factores) = suma de `n_blocks` ventanas de `block_length` cambios consecutivos, centradas.

    Cada inicio se sortea uniforme entre las T − b + 1 ventanas. Se centran las sumas de las ventanas (no los meses):
    sin vuelta circular, los meses de los extremos entran en menos ventanas, y centrar los meses deja un sesgo de
    media (D-09 v3). Así la media esperada del shock es exactamente 0. No incluye deriva.
    """
    x = changes.to_numpy(dtype=float)
    t = len(x)
    if not 1 <= block_length <= t:
        raise ValueError(f"block_length = {block_length} fuera de [1, {t}] (meses de historia)")
    if n < 1 or n_blocks < 1:
        raise ValueError("n y n_blocks deben ser ≥ 1")
    csum = np.vstack([np.zeros((1, x.shape[1])), np.cumsum(x, axis=0)])
    windows = csum[block_length:] - csum[:-block_length]  # (T − b + 1) × factores
    windows = windows - windows.mean(axis=0)
    starts = np.random.default_rng(seed).integers(0, len(windows), size=(n, n_blocks))
    return windows[starts].sum(axis=1)


def match_forward(log_shocks: np.ndarray, log_forward: float) -> np.ndarray:
    """Desplaza log-shocks equiprobables por una constante para que la media muestral de e^{shock} sea e^{log_forward}
    (D-24). Corrige a la vez la convexidad (E[e^X] = e^{μ + σ²/2}) y el error de muestreo de la media; la dispersión
    no cambia."""
    return log_shocks + (log_forward - math.log(float(np.mean(np.exp(log_shocks)))))


# ------------------------------------------------------------------------- estrés
def _isnull(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v))


_OVERRIDE_KEY = re.compile(r"^([a-z]+)_(\w+)_override$")


def _check_tenors(tenors: Mapping, node_years: Mapping[str, float]) -> None:
    """Los nodos de cada tramo y de su transición existen, no se repiten entre tramos y la transición no incluye t = 0."""
    seen: dict[str, str] = {}
    for bucket, spec in tenors.items():
        for node in list(spec.get("nodes", [])) + list(spec.get("transition", [])):
            if node not in node_years:
                raise ValueError(f"stressTenors.{bucket} usa el nodo inexistente {node}")
            if node in seen:
                raise ValueError(
                    f"El nodo {node} está en dos tramos de stressTenors ({seen[node]} y {bucket})"
                )
            seen[node] = bucket
        if any(node_years[n] <= 0 for n in spec.get("transition", [])):
            raise ValueError(f"stressTenors.{bucket}: un nodo de transición no puede estar en t = 0")


def _stress_curve_shift(
    parallel: float, overrides: Mapping[str, float], tenors: Mapping, node_years: Mapping[str, float]
) -> dict[str, float]:
    """Shift por nodo spot de una moneda (D-11 v2, D-16).

    Nodos fijos: el override en los nodos de un tramo con override y el paralelo en el resto, salvo las transiciones
    de tramos con override. En cada transición se interpola linealmente t·s(t) entre los nodos fijos vecinos: la
    forward implícita cambia lo mismo en todos los segmentos de la transición.
    """
    shift, pending = {}, []
    for node in node_years:
        bucket = next((b for b, spec in tenors.items() if node in spec.get("nodes", [])), None)
        trans = next((b for b, spec in tenors.items() if node in spec.get("transition", [])), None)
        if bucket in overrides:
            shift[node] = overrides[bucket]
        elif trans in overrides:
            pending.append(node)
        else:
            shift[node] = parallel
    for node in pending:
        t = node_years[node]
        left = max((n for n in shift if node_years[n] < t), key=node_years.get, default=None)
        right = min((n for n in shift if node_years[n] > t), key=node_years.get, default=None)
        if left is None or right is None:
            raise ValueError(f"El nodo de transición {node} necesita nodos fijos a ambos lados")
        tl, tr = node_years[left], node_years[right]
        acc = tl * shift[left] + (tr * shift[right] - tl * shift[left]) * (t - tl) / (tr - tl)
        shift[node] = acc / t
    return shift


def stress_shocks(doc: Mapping) -> pd.DataFrame:
    """Δ por escenario de estrés (escenarios × FACTORS), según D-11 v2 y D-16. Forwards en 0 (se recalculan).

    Exige `<ccy>_parallel` por moneda y rechaza `<ccy>_<tramo>_override` de tramos no definidos en `stressTenors`.
    """
    tenors = doc["caseParameters"]["stressTenors"]
    node_years = {n["id"]: float(n["years"]) for n in doc["curveNodes"]}
    _check_tenors(tenors, node_years)
    currencies = list(doc["marketHistory"][-1]["spot"])
    out = {}
    for s in doc["stressScenarios"]:
        sid = s["scenario_id"]
        for key in s:
            m = _OVERRIDE_KEY.match(key)
            if m and m.group(2) not in tenors:
                raise ValueError(
                    f"{sid}: {key} no corresponde a ningún tramo de stressTenors ({sorted(tenors)})"
                )
        d = dict.fromkeys(FACTORS, 0.0)
        for ccy in currencies:
            c = ccy.lower()
            if _isnull(s.get(f"{c}_parallel")):
                raise ValueError(f"{sid}: falta {c}_parallel")
            overrides = {
                b: float(s[f"{c}_{b}_override"]) for b in tenors if not _isnull(s.get(f"{c}_{b}_override"))
            }
            shift = _stress_curve_shift(float(s[f"{c}_parallel"]), overrides, tenors, node_years)
            d.update({f"{ccy}_SPOT_{n}": v for n, v in shift.items()})
        fx, eq = s.get("fx_pct"), s.get("equity_pct")
        d["FX_PENUSD"] = 0.0 if _isnull(fx) else math.log(1 + float(fx))
        d["EQUITY_USD"] = 0.0 if _isnull(eq) else math.log(1 + float(eq))
        out[sid] = d
    return pd.DataFrame.from_dict(out, orient="index")[FACTORS]


# ------------------------------------------------------------ conjunto de escenarios
@dataclass(frozen=True)
class ScenarioSet:
    """Escenarios a horizonte: ids, Δ de factores (S × FACTORS), probabilidades p_s y tipo (bootstrap/stress)."""

    ids: list[str]
    deltas: pd.DataFrame
    probs: np.ndarray
    kind: list[str]


def _horizon_months(doc: Mapping) -> int:
    t0, t_h = horizon_dates(doc)
    return (t_h.year - t0.year) * 12 + (t_h.month - t0.month)


def build_scenarios(doc: Mapping, seed: int | None = None) -> ScenarioSet:
    """Bootstrap (centrado + μ) y estrés, con p_s = bootstrapProbability / N_B y `probability` de cada estrés.

    `seed` reemplaza a `caseParameters.seed` (p. ej. para la prueba fuera de muestra).
    """
    cp = doc["caseParameters"]
    b, m = int(cp["bootstrapBlockLength"]), int(cp["bootstrapBlocksPerScenario"])
    months = _horizon_months(doc)
    if b * m != months:
        raise ValueError(
            f"bootstrapBlockLength × bootstrapBlocksPerScenario = {b * m} ≠ {months} meses de horizonte"
        )
    n_b = int(cp["bootstrapCount"])

    changes = factor_changes(doc)
    mu = drift(doc, changes).to_numpy()
    boot = bootstrap_shocks(changes, n_b, b, m, cp["seed"] if seed is None else seed)
    boot = boot + mu * (b * m / MONTHS_PER_YEAR)
    if cp["drift"]["fx"] == "IRP":
        i = FACTORS.index("FX_PENUSD")
        boot[:, i] = match_forward(boot[:, i], mu[i] * (b * m / MONTHS_PER_YEAR))
    stress = stress_shocks(doc)

    p_stress = np.array([float(s["probability"]) for s in doc["stressScenarios"]])
    if abs(p_stress.sum() - float(cp["stressProbability"])) > PROB_TOL:
        raise ValueError(
            f"Σ p de estrés = {p_stress.sum():.6f} ≠ stressProbability = {cp['stressProbability']}"
        )
    p_boot = np.full(n_b, float(cp["bootstrapProbability"]) / n_b)
    probs = np.concatenate([p_boot, p_stress])
    if abs(probs.sum() - 1.0) > PROB_TOL:
        raise ValueError(f"Las probabilidades de los escenarios suman {probs.sum():.6f}, no 1")

    width = len(str(n_b))
    ids = [f"B{i + 1:0{width}d}" for i in range(n_b)] + list(stress.index)
    deltas = pd.DataFrame(np.vstack([boot, stress.to_numpy()]), index=ids, columns=FACTORS)
    return ScenarioSet(ids, deltas, probs, ["bootstrap"] * n_b + ["stress"] * len(stress))


# ---------------------------------------------------------------- resultados r_{k,s}
def scenario_components(
    doc: Mapping, deltas: pd.DataFrame, spreads: Mapping[str, float] | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(reval, flows), instrumentos × escenarios, decimal por S/ invertido, por revalorización completa:
    reval = V^H/V^0 − 1 y flows = F/V^0 (flujos cobrados/pagados en (t0, tH], D-18). r = reval + flows."""
    check_positive_values(doc)
    spreads = calibrate_spreads(doc) if spreads is None else spreads
    v0 = pd.Series({i["id"]: float(i["market_value_pen"]) for i in doc["instruments"]})
    reval, flows = {}, {}
    for sid, row in deltas.iterrows():
        h = horizon_value_and_flows(doc, spreads, row.to_dict())
        base = v0.reindex(h.index)
        reval[sid] = h["value_h"] / base - 1
        flows[sid] = h["flows"] / base
    return pd.DataFrame(reval), pd.DataFrame(flows)


def scenario_returns(
    doc: Mapping, deltas: pd.DataFrame, spreads: Mapping[str, float] | None = None
) -> pd.DataFrame:
    """r_{k,s} (instrumentos × escenarios, decimal por S/ invertido) por revalorización completa al horizonte."""
    reval, flows = scenario_components(doc, deltas, spreads)
    return reval + flows


def pn_change(returns: pd.DataFrame, positions: pd.Series, sides: pd.Series) -> np.ndarray:
    """ΔPN_s = Σ_i x_i r_{i,s} − Σ_j y_j r_{j,s} (S/ mm), con `sides` ∈ {ASSET, LIABILITY}."""
    bad = set(sides.reindex(positions.index)) - {"ASSET", "LIABILITY"}
    if bad:
        raise ValueError(f"Lados desconocidos: {bad}")
    sign = np.where(sides.reindex(positions.index) == "ASSET", 1.0, -1.0)
    return (sign * positions.to_numpy(dtype=float)) @ returns.loc[positions.index].to_numpy(dtype=float)


# ------------------------------------------------------------------------ VaR / CVaR
def var_cvar(losses: np.ndarray, alpha: float, probs: np.ndarray | None = None) -> tuple[float, float]:
    """VaR_α = min{ℓ : P(L ≤ ℓ) ≥ α} y CVaR_α = [Σ_{L>VaR} p L + VaR (1 − α − Σ_{L>VaR} p)] / (1 − α).

    Pérdidas en S/ mm (pérdida > 0). Probabilidades uniformes si `probs` es None.
    """
    if not 0 < alpha < 1:
        raise ValueError(f"alpha = {alpha} debe estar en (0, 1)")
    losses = np.asarray(losses, dtype=float)
    probs = np.full(len(losses), 1 / len(losses)) if probs is None else np.asarray(probs, dtype=float)
    if probs.shape != losses.shape or (probs < 0).any() or abs(probs.sum() - 1) > PROB_TOL:
        raise ValueError("probs debe tener la forma de losses, ser ≥ 0 y sumar 1")
    order = np.argsort(losses, kind="stable")
    l_sorted, cum = losses[order], np.cumsum(probs[order])
    var = float(l_sorted[np.searchsorted(cum, alpha - 1e-12)])
    tail = losses > var
    p_tail = probs[tail].sum()
    cvar = (probs[tail] @ losses[tail] + var * (1 - alpha - p_tail)) / (1 - alpha)
    return var, float(cvar)


# ------------------------------------------------------------------------- controles
def scenario_diagnostics(doc: Mapping, scenarios: ScenarioSet) -> dict[str, pd.Series | pd.DataFrame]:
    """Controles de los escenarios.

    - sigma: σ del shock bootstrap vs. σ histórica de la suma móvil de b·m meses (sin centrar), por factor.
    - min_rate: tasa spot mínima (nodos) de cada escenario al horizonte, sin piso (§7.3).
    """
    cp = doc["caseParameters"]
    window = int(cp["bootstrapBlockLength"]) * int(cp["bootstrapBlocksPerScenario"])
    changes = factor_changes(doc)
    boot = scenarios.deltas[np.asarray(scenarios.kind) == "bootstrap"]
    sigma = pd.DataFrame(
        {"bootstrap_12m": boot.std(ddof=1), "historical_12m": changes.rolling(window).sum().std(ddof=1)}
    )
    t0_row = history_from_doc(doc).loc[horizon_dates(doc)[0], SPOT_FACTORS]
    min_rate = (scenarios.deltas[SPOT_FACTORS] + t0_row).min(axis=1)
    return {"sigma": sigma, "min_rate": min_rate}


# ------------------------------------------------------------ chequeo local (D-14)
def sensitivities(
    value_fn: Callable[[Mapping[str, float]], pd.Series],
    factors: Sequence[str],
    bump_rate: float = 1e-4,
    bump_log: float = 0.01,
) -> pd.DataFrame:
    """Matriz M (instrumentos × factores) por diferencias finitas centrales: ∂V/∂F en S/ mm por unidad del factor
    (1.0 = 100 % en tasas; 1.0 de log-retorno en FX/equity). Bump de 1 pb en tasas y 1 % en log-factores.

    Solo para el chequeo de coherencia local vs. revalorización completa (D-14), no para el modelo.
    """
    cols = {}
    for f in factors:
        h = bump_log if f in LOG_FACTORS else bump_rate
        cols[f] = (value_fn({f: h}) - value_fn({f: -h})) / (2 * h)
    return pd.DataFrame(cols)


def covariance(changes: pd.DataFrame, annualize: int = MONTHS_PER_YEAR) -> pd.DataFrame:
    """Covarianza muestral de cambios mensuales, anualizada (×12, i.i.d.) por defecto."""
    return changes.cov() * annualize


def nearest_psd(m: np.ndarray, eps: float = 0.0) -> np.ndarray:
    """Proyección a semidefinida positiva por recorte de autovalores."""
    m = (m + m.T) / 2
    w, v = np.linalg.eigh(m)
    return (v * np.clip(w, eps, None)) @ v.T

### (iv) Optimizador

(iv) Formulación y resolución del Caso C: max E[ΔPN] − λ·CVaR_α − TC (docs/formulacion.md §4-§6, §9).

Regla de oro: todo sale del dict de input.json. Ningún límite, λ, α, costo o presupuesto está escrito aquí.

- LP de Rockafellar–Uryasev con cvxpy (HiGHS); benchmark media-varianza como QP (Clarabel).
- r_{k,s} = reval + flows por revalorización completa (risk.scenario_components); se calcula una vez y se reutiliza
  en el barrido de λ (D-19).
- VaR y CVaR se recalculan ex post con átomo fraccional (risk.var_cvar), §5.
- Caja al horizonte = instrumentos CASH + flujos netos del año − TC (D-18). Los incumplimientos al horizonte se
  reportan, no se imponen (§7.5).
- Montos en S/ mm; pesos y tasas en decimales.

In [ ]:
%%writefile {LIB}/optimum/optimizer.py
"""(iv) Formulación y resolución del Caso C: max E[ΔPN] − λ·CVaR_α − TC (docs/formulacion.md §4-§6, §9).

Regla de oro: todo sale del dict de input.json. Ningún límite, λ, α, costo o presupuesto está escrito aquí.

- LP de Rockafellar–Uryasev con cvxpy (HiGHS); benchmark media-varianza como QP (Clarabel).
- r_{k,s} = reval + flows por revalorización completa (risk.scenario_components); se calcula una vez y se reutiliza
  en el barrido de λ (D-19).
- VaR y CVaR se recalculan ex post con átomo fraccional (risk.var_cvar), §5.
- Caja al horizonte = instrumentos CASH + flujos netos del año − TC (D-18). Los incumplimientos al horizonte se
  reportan, no se imponen (§7.5).
- Montos en S/ mm; pesos y tasas en decimales.
"""

from __future__ import annotations

import re
from collections.abc import Mapping
from dataclasses import dataclass, field

import cvxpy as cp
import numpy as np
import pandas as pd

from optimum import risk
from optimum.io.json_contract import SCHEMA_VERSION
from optimum.valuation import horizon_dates

LP_SOLVER = cp.HIGHS
QP_SOLVER = cp.CLARABEL
OK_TOL = 1e-6  # tolerancia relativa para declarar cumplida una restricción
ACTIVE_TOL = 1e-7  # tolerancia relativa para declararla activa (en su cota)
OPTIMAL = {"OPTIMAL", "OPTIMAL_INACCURATE"}
HORIZON_MATURITY_MODES = ("REPORT", "ENFORCE")  # D-20
_STATUS = {
    cp.OPTIMAL: "OPTIMAL",
    cp.OPTIMAL_INACCURATE: "OPTIMAL_INACCURATE",
    cp.INFEASIBLE: "INFEASIBLE",
    cp.INFEASIBLE_INACCURATE: "INFEASIBLE",
    cp.UNBOUNDED: "UNBOUNDED",
    cp.UNBOUNDED_INACCURATE: "UNBOUNDED",
}
# Categorías de TYPE que agrupan varios `instrument_type` (§6); el resto se compara por igualdad.
TYPE_MEMBERS = {"FIXED_INCOME": {"BOND_FIXED", "BOND_FLOAT"}}
_MATURITY = re.compile(r"^(LE|LT|GE|GT)_(\d+)M$")


# ------------------------------------------------------------------------------ datos
@dataclass
class ProblemData:
    """Datos del problema, todos leídos del doc (y de los componentes de escenario ya revalorizados)."""

    ids: list[str]
    side: pd.Series  # ASSET / LIABILITY
    x0: pd.Series  # posición inicial, S/ mm
    reval: pd.DataFrame  # K × S, V^H/V^0 − 1
    flows: pd.DataFrame  # K × S, F/V^0
    probs: np.ndarray
    scenario_ids: list[str]
    scenario_kind: list[str]
    cost: pd.Series  # restructuring_cost por S/ movido (D-08)
    alpha: float
    lam: float
    groups: list[dict]  # restricciones de peso en t0
    horizon_groups: list[dict]  # los mismos grupos con la pertenencia medida desde tH (D-20)
    asset_budget: float
    liability_budget: float
    min_net_worth: float
    max_liabilities_to_assets: float
    min_cash: float
    allow_short: bool
    cash_ids: list[str]
    horizon_maturity: str  # REPORT | ENFORCE (D-20)
    base_currency: str
    instruments: pd.DataFrame = field(repr=False)
    lambda_grid: list[float] = field(default_factory=list)

    @property
    def returns(self) -> pd.DataFrame:
        """r_{k,s} = reval + flows (decimal por S/ invertido)."""
        return self.reval + self.flows

    @property
    def sign(self) -> np.ndarray:
        """+1 activos, −1 pasivos (ΔPN = Σ x r − Σ y r)."""
        return np.where(self.side.loc[self.ids] == "ASSET", 1.0, -1.0)

    def budget(self, side: str) -> float:
        return self.asset_budget if side == "ASSET" else self.liability_budget


def _is_member(inst: pd.Series, dimension: str, category: str, t_ref: pd.Timestamp) -> bool:
    """Pertenencia a un grupo de pesos por atributos del instrumento (§6), nunca por id."""
    if dimension == "CURRENCY":
        return inst["currency"] == category
    if dimension == "TYPE":
        return inst["instrument_type"] in TYPE_MEMBERS.get(category, {category})
    if dimension == "RATE_TYPE":
        floating = isinstance(inst["reference_factor"], str) and bool(inst["reference_factor"])
        if category not in ("FIXED", "FLOAT"):
            raise ValueError(f"Categoría RATE_TYPE desconocida: {category}")
        return floating == (category == "FLOAT")
    if dimension == "MATURITY":
        m = _MATURITY.match(category)
        if not m:
            raise ValueError(f"Categoría MATURITY desconocida: {category} (se espera LE_12M, GT_36M, ...)")
        if inst["maturity"] is None or pd.isna(inst["maturity"]):
            return False
        # Meses calendario desde t_ref (auditoría, menor 1): 2026-12-31 + 36 m = 2029-12-31.
        op, maturity = m.group(1), pd.Timestamp(inst["maturity"])
        limit = t_ref + pd.DateOffset(months=int(m.group(2)))
        return {"LE": maturity <= limit, "LT": maturity < limit, "GE": maturity >= limit, "GT": maturity > limit}[op]
    raise ValueError(f"Dimensión de peso desconocida: {dimension}")


def _build_groups(
    weights: list[dict], inst: pd.DataFrame, t_ref: pd.Timestamp, check_empty: bool
) -> list[dict]:
    groups = []
    for w in weights:
        side, dim, cat = w["side"], w["dimension"], w["category"]
        on_side = inst.index[inst["side"] == side]
        base = {
            "side": side,
            "dimension": dim,
            "category": cat,
            "min": w["min_weight"],
            "max": w["max_weight"],
        }
        if dim == "CONCENTRATION":
            for iid in on_side:
                groups.append({**base, "name": f"{side}:{dim}:{cat}:{iid}", "members": [iid]})
            continue
        members = [iid for iid in on_side if _is_member(inst.loc[iid], dim, cat, t_ref)]
        if check_empty and not members and float(w["min_weight"] or 0.0) > 0:
            raise ValueError(
                f"El grupo de pesos {side}/{dim}/{cat} no tiene instrumentos pero exige "
                f"min_weight = {w['min_weight']}: el problema no tiene sentido con estos datos"
            )
        groups.append({**base, "name": f"{side}:{dim}:{cat}", "members": members})
    return groups


def _validate_parameters(doc: Mapping, inst: pd.DataFrame, weights: list[dict]) -> None:
    """Parámetros inválidos fallan con un mensaje claro (auditoría, menores 2 y 5)."""
    cp_ = doc["caseParameters"]
    errors = []
    bad_cost = [i for i in inst.index if float(doc["transactionCosts"][i]["restructuring_cost"]) < 0]
    if bad_cost:
        errors.append(f"restructuring_cost < 0 en {bad_cost}")
    if not 0 < float(cp_["cvarAlpha"]) < 1:
        errors.append(f"cvarAlpha = {cp_['cvarAlpha']} debe estar en (0, 1)")
    lams = [float(cp_["riskAversionLambda"])] + [float(v) for v in cp_.get("lambdaGrid", [])]
    if min(lams) < 0:
        errors.append("riskAversionLambda y lambdaGrid deben ser ≥ 0")
    mode = cp_.get("horizonMaturityLimits", "REPORT")
    if mode not in HORIZON_MATURITY_MODES:
        errors.append(f"horizonMaturityLimits = {mode!r}; se espera uno de {HORIZON_MATURITY_MODES}")
    for side in ("ASSET", "LIABILITY"):
        cats = [w["category"] for w in weights if w["side"] == side and w["dimension"] == "TYPE"]
        if not cats:
            continue
        for iid in inst.index[inst["side"] == side]:
            n = sum(_is_member(inst.loc[iid], "TYPE", c, pd.Timestamp(doc["valuationDate"])) for c in cats)
            if n != 1:
                errors.append(
                    f"{iid} ({inst.loc[iid, 'instrument_type']}) pertenece a {n} categorías TYPE de {side}; "
                    "debe pertenecer a exactamente una"
                )
    if errors:
        raise ValueError("Parámetros inválidos:\n- " + "\n- ".join(errors))


def prepare(doc: Mapping, components: tuple | None = None) -> ProblemData:
    """Arma ProblemData desde el doc. `components` = (reval, flows, ScenarioSet) evita revalorizar (~0.1 s por
    escenario); si es None se generan con `risk.build_scenarios` y `risk.scenario_components`."""
    if components is None:
        scen = risk.build_scenarios(doc)
        reval, flows = risk.scenario_components(doc, scen.deltas)
    else:
        reval, flows, scen = components
    inst = pd.DataFrame(doc["instruments"]).set_index("id")
    ids = list(inst.index)
    t0, t_h = horizon_dates(doc)
    missing = [i for i in ids if i not in reval.index or i not in flows.index]
    if missing:
        raise ValueError(f"Faltan resultados por escenario para: {missing}")
    costs = doc["transactionCosts"]
    no_cost = [i for i in ids if i not in costs]
    if no_cost:
        raise ValueError(f"transactionCosts no trae restructuring_cost para: {no_cost}")

    cp_ = doc["caseParameters"]
    com = doc["constraints"]["common"]
    weights = doc["constraints"]["weights"]
    _validate_parameters(doc, inst, weights)
    return ProblemData(
        ids=ids,
        side=inst["side"],
        x0=inst["market_value_pen"].astype(float),
        reval=reval.loc[ids, list(scen.ids)],
        flows=flows.loc[ids, list(scen.ids)],
        probs=np.asarray(scen.probs, dtype=float),
        scenario_ids=list(scen.ids),
        scenario_kind=list(scen.kind),
        cost=pd.Series({i: float(costs[i]["restructuring_cost"]) for i in ids}),
        alpha=float(cp_["cvarAlpha"]),
        lam=float(cp_["riskAversionLambda"]),
        groups=_build_groups(weights, inst, t0, check_empty=True),
        horizon_groups=_build_groups(weights, inst, t_h, check_empty=False),
        asset_budget=float(com["assetBudget"]),
        liability_budget=float(com["liabilityBudget"]),
        min_net_worth=float(com["minNetWorth"]),
        max_liabilities_to_assets=float(com["maxLiabilitiesToAssets"]),
        min_cash=float(com["minCash"]),
        allow_short=bool(com.get("allowShort", False)),
        cash_ids=list(inst.index[(inst["side"] == "ASSET") & (inst["instrument_type"] == "CASH")]),
        horizon_maturity=cp_.get("horizonMaturityLimits", "REPORT"),
        base_currency=doc.get("baseCurrency", "PEN"),
        instruments=inst,
        lambda_grid=[float(v) for v in cp_.get("lambdaGrid", [])],
    )


# ------------------------------------------------------------------------ modelo
def _mask(data: ProblemData, members) -> np.ndarray:
    s = set(members)
    return np.array([i in s for i in data.ids], dtype=float)


def _horizon_maturity_groups(data: ProblemData) -> list[dict]:
    return [g for g in data.horizon_groups if g["dimension"] == "MATURITY"]


def _feasible_set(data: ProblemData, z: cp.Variable, horizon_maturity: str) -> list[tuple[str, object, float]]:
    """Presupuestos, pesos, caja mínima, PN0 y P/A0 (§6), como (nombre, restricción, escala del dual). Las
    posiciones ≥ 0 van en el dominio de z. Con ENFORCE se imponen además los grupos MATURITY con la pertenencia
    medida desde tH (D-20; nombre con sufijo @tH).

    Escala: dual × escala = ganancia de objetivo (S/ mm) por unidad de relajación del parámetro del input (1.0 de
    peso en los grupos; 1 S/ mm en montos; 1.0 de ratio en P/A). En los presupuestos (igualdades) el signo es el de
    subir el presupuesto."""
    a = _mask(data, [i for i in data.ids if data.side[i] == "ASSET"])
    assets, liabs = a @ z, (1.0 - a) @ z
    cons = [
        ("assetBudget", assets == data.asset_budget, 1.0),
        ("liabilityBudget", liabs == data.liability_budget, 1.0),
    ]
    enforced = data.groups + (
        [{**g, "name": f"{g['name']}@tH"} for g in _horizon_maturity_groups(data)]
        if horizon_maturity == "ENFORCE"
        else []
    )
    for g in enforced:
        total = _mask(data, g["members"]) @ z
        b = data.budget(g["side"])
        cons += [(f"{g['name']}:min", total >= g["min"] * b, b), (f"{g['name']}:max", total <= g["max"] * b, b)]
    cons += [
        ("minCash", _mask(data, data.cash_ids) @ z >= data.min_cash, 1.0),
        ("minNetWorth", assets - liabs >= data.min_net_worth, 1.0),
        ("maxLiabilitiesToAssets", liabs <= data.max_liabilities_to_assets * assets, data.asset_budget),
    ]
    return cons


def solve_data(
    data: ProblemData,
    lam: float | None = None,
    mode: str = "cvar",
    min_net_return: float | None = None,
    horizon_maturity: str | None = None,
    duals: bool = False,
) -> dict:
    """Resuelve el problema. mode="cvar": max E[ΔPN] − λ·CVaR − TC (LP, λ = data.lam si es None).
    mode="variance": min Var(ΔPN) s.a. E[ΔPN] − TC ≥ min_net_return (QP, §9.3).

    Devuelve status, positions (pd.Series S/ mm, None si no es óptimo), objective, cvar_lp (ζ + Σ p u/(1−α),
    solo con λ > 0), transaction_cost y solver. Con `duals=True` agrega `duals`: {nombre: ganancia de objetivo por
    unidad de relajación} de cada restricción de `_feasible_set` (S2, formulación §11). Nunca lanza por
    infactibilidad.
    """
    lam = data.lam if lam is None else float(lam)
    horizon_maturity = horizon_maturity or data.horizon_maturity
    k, s = len(data.ids), len(data.scenario_ids)
    r = data.returns.to_numpy(dtype=float)  # K × S
    z = cp.Variable(k, nonneg=not data.allow_short)
    up, down = cp.Variable(k, nonneg=True), cp.Variable(k, nonneg=True)
    dpn = r.T @ cp.multiply(data.sign, z)
    expected = data.probs @ dpn
    tc = data.cost.loc[data.ids].to_numpy() @ (up + down)
    named = _feasible_set(data, z, horizon_maturity)
    cons = [c for _, c, _ in named] + [z - data.x0.loc[data.ids].to_numpy() == up - down]

    cvar = None
    if mode == "cvar":
        zeta, u = cp.Variable(), cp.Variable(s, nonneg=True)
        cons.append(u >= -dpn - zeta)
        cvar = zeta + data.probs @ u / (1 - data.alpha)
        problem, solver = cp.Problem(cp.Maximize(expected - lam * cvar - tc), cons), LP_SOLVER
    elif mode == "variance":
        if min_net_return is None:
            raise ValueError("mode='variance' requiere min_net_return (E* de la solución CVaR)")
        dev = cp.multiply(np.sqrt(data.probs), dpn - expected)
        cons.append(expected - tc >= min_net_return)
        problem, solver = cp.Problem(cp.Minimize(cp.sum_squares(dev)), cons), QP_SOLVER
    else:
        raise ValueError(f"mode desconocido: {mode}")

    try:
        problem.solve(solver=solver)
        status = _STATUS.get(problem.status, "ERROR")
    except cp.error.SolverError:
        status = "ERROR"
    stats = problem.solver_stats if problem.solver_stats is not None else None
    out = {
        "status": status,
        "mode": mode,
        "lambda": lam,
        "horizon_maturity": horizon_maturity,
        "positions": None,
        "objective": None,
        "cvar_lp": None,
        "transaction_cost": None,
        "solver": {
            "name": solver,
            "status": problem.status,
            "iterations": getattr(stats, "num_iters", None),
            "time": getattr(stats, "solve_time", None),
        },
    }
    if status not in OPTIMAL:
        return out
    pos = pd.Series(z.value, index=data.ids)
    out["positions"] = pos
    out["transaction_cost"] = _transaction_cost(data, pos)
    out["objective"] = float(problem.value)
    if cvar is not None and lam > 0:
        out["cvar_lp"] = float(cvar.value)
    if duals:
        out["duals"] = {name: _dual_gain(c) * scale for name, c, scale in named}
    return out


def _dual_gain(con) -> float:
    """Dual de cvxpy como ganancia del objetivo (Maximize) por unidad de relajación de la restricción. En las
    desigualdades cvxpy entrega μ ≥ 0; en las igualdades, el signo corresponde a subir el lado derecho."""
    return float(np.asarray(con.dual_value).ravel()[0])


# --------------------------------------------------------------------- evaluación
def _transaction_cost(data: ProblemData, pos: pd.Series) -> float:
    return float(data.cost.loc[data.ids] @ (pos.loc[data.ids] - data.x0.loc[data.ids]).abs())


def _positions(data: ProblemData, positions: pd.Series) -> pd.Series:
    pos = pd.Series(positions, dtype=float).reindex(data.ids)
    if pos.isna().any():
        raise ValueError(f"Faltan posiciones para: {list(pos.index[pos.isna()])}")
    return pos


def _tail_stress_share(data: ProblemData, losses: np.ndarray) -> float:
    """Fracción de la masa de la cola (1 − α) que aportan los escenarios de estrés (átomo fraccional incluido)."""
    order = np.argsort(-losses, kind="stable")
    left, stress = 1 - data.alpha, 0.0
    for i in order:
        take = min(data.probs[i], left)
        if data.scenario_kind[i] == "stress":
            stress += take
        left -= take
        if left <= 1e-15:
            break
    return stress / (1 - data.alpha)


def _breach(mask: np.ndarray, probs: np.ndarray, worst: float) -> dict:
    return {"count": int(mask.sum()), "probability": float(probs[mask].sum()), "worst": float(worst)}


def _horizon_weight_breaches(
    data: ProblemData, z: np.ndarray, value_h: np.ndarray, net_cash_in: np.ndarray
) -> dict:
    """Pesos del Cuadro 4 al horizonte, por escenario (auditoría I-2, D-20).

    Tenencias en tH: V^H de cada instrumento; los flujos netos del año menos el TC (`net_cash_in`) se suman a la
    caja en moneda base (a prorrata si hay varias; a toda la caja si no hay en moneda base). La pertenencia se
    mide desde tH. Causa "estructural": el grupo se incumple aun con los montos de t0 (lo provoca el paso del
    tiempo sobre la decisión); "mercado": solo por la deriva de valores.
    """
    held = value_h.copy()
    target = [i for i in data.cash_ids if data.instruments.loc[i, "currency"] == data.base_currency] or data.cash_ids
    if target:
        idx = [data.ids.index(i) for i in target]
        share = z[idx] / z[idx].sum() if z[idx].sum() > 0 else np.full(len(idx), 1 / len(idx))
        held[idx] += share[:, None] * net_cash_in
    out = {}
    for g in data.horizon_groups:
        on_side = (data.side.loc[data.ids] == g["side"]).to_numpy()
        m = _mask(data, g["members"]).astype(bool)
        total = held[on_side].sum(0)
        w = np.divide(held[m].sum(0), total, out=np.zeros_like(total), where=total != 0)
        excess = np.maximum(g["min"] - w, w - g["max"])
        bad = excess > OK_TOL
        w0 = z[m].sum() / z[on_side].sum() if z[on_side].sum() else 0.0
        structural = w0 < g["min"] - OK_TOL or w0 > g["max"] + OK_TOL
        out[g["name"]] = {
            "count": int(bad.sum()),
            "probability": float(data.probs[bad].sum()),
            "worst": float(w[np.argmax(excess)]),
            "range": [float(w.min()), float(w.max())],
            "min": g["min"],
            "max": g["max"],
            "cause": None if not bad.any() else ("estructural" if structural else "mercado"),
        }
    return out


def evaluate(data: ProblemData, positions: pd.Series) -> dict:
    """Métricas ex post de una cartera (S/ mm): E[ΔPN], VaR/CVaR (átomo fraccional), volatilidad (desviación
    estándar ponderada por p_s), pérdida en cada estrés, peso del estrés en la cola, TC e incumplimientos al
    horizonte (§7.5, D-18, D-20): PN, P/A, caja y todos los grupos del Cuadro 4."""
    pos = _positions(data, positions)
    z = pos.to_numpy()
    reval, flows = data.reval.to_numpy(dtype=float), data.flows.to_numpy(dtype=float)
    dpn = (data.sign * z) @ (reval + flows)
    losses = -dpn
    var, cvar = risk.var_cvar(losses, data.alpha, data.probs)
    expected = float(data.probs @ dpn)
    tc = _transaction_cost(data, pos)
    is_asset = (data.side.loc[data.ids] == "ASSET").to_numpy()
    is_cash = _mask(data, data.cash_ids).astype(bool)
    mean_r = (reval + flows) @ data.probs

    # Balance al horizonte (D-18): los flujos del año y el TC pagado en t0 quedan en caja.
    value_h = z[:, None] * (1 + reval)
    flow_h = z[:, None] * flows
    cash_t = value_h[is_cash].sum(0) + flow_h[is_asset].sum(0) - flow_h[~is_asset].sum(0) - tc
    assets_t = value_h[is_asset & ~is_cash].sum(0) + cash_t
    liabs_t = value_h[~is_asset].sum(0)
    nw_t = assets_t - liabs_t
    ratio_t = liabs_t / assets_t
    breaches = {
        "minNetWorth": _breach(nw_t < data.min_net_worth - OK_TOL, data.probs, nw_t.min()),
        "maxLiabilitiesToAssets": _breach(
            ratio_t > data.max_liabilities_to_assets + OK_TOL, data.probs, ratio_t.max()
        ),
        "minCash": _breach(cash_t < data.min_cash - OK_TOL, data.probs, cash_t.min()),
    }
    breaches |= _horizon_weight_breaches(data, z, value_h, cash_t - value_h[is_cash].sum(0))

    a_total, l_total = z[is_asset].sum(), z[~is_asset].sum()
    return {
        "expected_pnl": expected,
        "var": var,
        "cvar": cvar,
        "volatility": float(np.sqrt(data.probs @ (dpn - expected) ** 2)),
        "stress_losses": {
            sid: float(losses[i])
            for i, sid in enumerate(data.scenario_ids)
            if data.scenario_kind[i] == "stress"
        },
        "tail_stress_share": _tail_stress_share(data, losses),
        "transaction_cost": tc,
        "initial_net_worth": float(a_total - l_total),
        "expected_terminal_net_worth": float(a_total - l_total - tc + expected),
        "expected_asset_return": float(z[is_asset] @ mean_r[is_asset] / a_total) if a_total else None,
        "expected_liability_cost": float(z[~is_asset] @ mean_r[~is_asset] / l_total) if l_total else None,
        "horizon_breaches": breaches,
    }


def _check(
    name: str, value: float, lo: float | None, hi: float | None, binding: bool = True, note: str = ""
) -> dict:
    def near(b, tol):
        return b is not None and abs(value - b) <= tol * max(1.0, abs(b))

    ok = (lo is None or value >= lo - OK_TOL * max(1.0, abs(lo))) and (
        hi is None or value <= hi + OK_TOL * max(1.0, abs(hi))
    )
    active = binding and (near(lo, ACTIVE_TOL) or near(hi, ACTIVE_TOL))
    return {
        "name": name,
        "value": float(value),
        "min": lo,
        "max": hi,
        "active": bool(active),
        "ok": bool(ok),
        "note": note,
    }


def check_constraints(data: ProblemData, positions: pd.Series) -> list[dict]:
    """[{name, value, min, max, active, ok, note}] de cada restricción de §6. Pesos como fracción del total del lado;
    montos en S/ mm. Los grupos vacíos se marcan como no vinculantes."""
    pos = _positions(data, positions)
    is_asset = data.side.loc[data.ids] == "ASSET"
    assets, liabs = float(pos[is_asset].sum()), float(pos[~is_asset].sum())
    checks = [
        _check("assetBudget", assets, data.asset_budget, data.asset_budget),
        _check("liabilityBudget", liabs, data.liability_budget, data.liability_budget),
    ]
    for g in data.groups:
        total = assets if g["side"] == "ASSET" else liabs
        w = float(pos.loc[g["members"]].sum() / total) if total else 0.0
        empty = not g["members"]
        checks.append(
            _check(g["name"], w, g["min"], g["max"], binding=not empty, note="grupo vacío" if empty else "")
        )
    if data.horizon_maturity == "ENFORCE":
        for g in _horizon_maturity_groups(data):
            total = assets if g["side"] == "ASSET" else liabs
            w = float(pos.loc[g["members"]].sum() / total) if total else 0.0
            checks.append(_check(f"{g['name']}@tH", w, g["min"], g["max"], binding=bool(g["members"])))
    checks += [
        _check("minCash", float(pos.loc[data.cash_ids].sum()), data.min_cash, None),
        _check("minNetWorth", assets - liabs, data.min_net_worth, None),
        _check(
            "maxLiabilitiesToAssets",
            liabs / assets if assets else np.inf,
            None,
            data.max_liabilities_to_assets,
        ),
    ]
    if not data.allow_short:
        checks.append(_check("nonNegativity", float(pos.min()), 0.0, None))
    return checks


# ------------------------------------------------------------------------ output
def _metrics_block(ev: dict, alpha: float) -> dict:
    tag = f"{alpha * 100:g}"
    return {
        "expectedAssetReturn": ev["expected_asset_return"],
        "expectedLiabilityEconomicCost": ev["expected_liability_cost"],
        "expectedNetWorthChange": ev["expected_pnl"],
        "expectedTerminalNetWorth": ev["expected_terminal_net_worth"],
        "netWorthVolatility": ev["volatility"],
        f"var{tag}Loss": ev["var"],
        f"cvar{tag}Loss": ev["cvar"],
        "cvarAlpha": alpha,
        "transactionCosts": ev["transaction_cost"],
        "stressLosses": ev["stress_losses"],
        "tailStressShare": ev["tail_stress_share"],
        "horizonBreaches": ev["horizon_breaches"],
    }


def _summary(data: ProblemData, res: dict, lam: float) -> dict:
    """Resumen de una solución para el bloque `analysis`."""
    if res["status"] not in OPTIMAL:
        return {"lambda": lam, "status": res["status"]}
    ev = evaluate(data, res["positions"])
    return {
        "lambda": lam,
        "status": res["status"],
        "objective": ev["expected_pnl"] - lam * ev["cvar"] - ev["transaction_cost"],
        "metrics": _metrics_block(ev, data.alpha),
        "weights": _weights(data, res["positions"]),
        "positions": {k: float(v) for k, v in res["positions"].items()},
    }


def _weights(data: ProblemData, pos: pd.Series) -> dict:
    """Pesos por lado y dimensión (fracción del total del lado)."""
    inst = data.instruments.loc[data.ids]
    rate_type = np.where(
        inst["reference_factor"].map(lambda v: isinstance(v, str) and bool(v)), "FLOAT", "FIXED"
    )
    out = {}
    for label, key in (
        ("byCurrency", inst["currency"]),
        ("byType", inst["instrument_type"]),
        ("byRateType", rate_type),
    ):
        out[label] = {}
        for side in ("ASSET", "LIABILITY"):
            m = (inst["side"] == side).to_numpy()
            total = float(pos[m].sum())
            grouped = pd.Series(pos[m].to_numpy(), index=np.asarray(key)[m]).groupby(level=0).sum()
            out[label][side] = {k: float(v / total) for k, v in grouped.items()} if total else {}
    return out


def _position_rows(data: ProblemData, pos: pd.Series | None, side: str) -> list[dict]:
    inst = data.instruments
    rows = []
    for iid in [i for i in data.ids if data.side[i] == side]:
        x = None if pos is None else float(pos[iid])
        rows.append(
            {
                "id": iid,
                "name": inst.loc[iid, "name"],
                "type": inst.loc[iid, "instrument_type"],
                "currency": inst.loc[iid, "currency"],
                "initial": float(data.x0[iid]),
                "optimal": x,
                "change": None if x is None else x - float(data.x0[iid]),
                "weight": None if x is None else x / data.budget(side),
            }
        )
    return rows


def _risk_model(doc: Mapping, data: ProblemData) -> dict:
    hist = risk.history_from_doc(doc)
    r = data.returns.to_numpy(dtype=float)
    mean = r @ data.probs
    cov = ((r - mean[:, None]) * data.probs) @ (r - mean[:, None]).T
    min_eig = float(np.linalg.eigvalsh(cov).min())
    cp_ = doc["caseParameters"]
    return {
        "historyStart": hist.index.min(),
        "historyEnd": hist.index.max(),
        "observations": len(hist),
        "covarianceMethod": "Sin covarianza paramétrica: escenarios (bootstrap por bloques + estrés) con "
        "revalorización completa; covarianza de r_{k,s} ponderada por p_s solo como control",
        "psdCheck": {"ok": min_eig >= -1e-12, "minEigenvalue": min_eig},
        "riskMeasure": cp_.get("riskMeasure"),
        "cvarAlpha": data.alpha,
        "scenarios": len(data.scenario_ids),
        "bootstrapScenarios": data.scenario_kind.count("bootstrap"),
        "stressScenarios": data.scenario_kind.count("stress"),
        "stressProbability": float(data.probs[np.array(data.scenario_kind) == "stress"].sum()),
        "seed": cp_.get("seed"),
    }


def _horizon_variant(data: ProblemData, base: dict) -> dict:
    """La misma λ con el modo de vencimientos opuesto al del input (D-20). `objectiveCost` = objetivo base −
    objetivo de la variante: con base REPORT es el costo de cumplir los vencimientos medidos desde tH."""
    mode = "ENFORCE" if data.horizon_maturity == "REPORT" else "REPORT"
    alt = solve_data(data, horizon_maturity=mode)
    summary = {"mode": mode, **_summary(data, alt, data.lam)}
    if alt["status"] in OPTIMAL:
        summary["objectiveCost"] = base["objective"] - summary["objective"]
    return summary


def solve(doc: Mapping, components: tuple | None = None) -> dict:
    """Output v1.0 (Anexo 1.B) de la solución con λ = riskAversionLambda, más `analysis` (D-19): posición inicial,
    barrido de `lambdaGrid` y benchmark media-varianza con el mismo retorno neto. Si no es óptimo, devuelve el
    status sin posiciones ni métricas (no lanza). Incluye la variante de vencimientos al horizonte (D-20)."""
    data = prepare(doc, components)
    res = solve_data(data)
    t0, t_h = horizon_dates(doc)
    is_asset = data.side.loc[data.ids] == "ASSET"
    out = {
        "schemaVersion": SCHEMA_VERSION,
        "caseId": doc["caseId"],
        "status": res["status"],
        "objectiveValue": res["objective"],
        "valuation": {
            "valuationDate": t0,
            "horizonDate": t_h,
            "initialAssets": float(data.x0[is_asset].sum()),
            "initialLiabilities": float(data.x0[~is_asset].sum()),
            "initialNetWorth": float(data.x0[is_asset].sum() - data.x0[~is_asset].sum()),
        },
        "metrics": {},
        "riskModel": _risk_model(doc, data),
        "positions": {
            "assets": _position_rows(data, res["positions"], "ASSET"),
            "liabilities": _position_rows(data, res["positions"], "LIABILITY"),
        },
        "weights": {},
        "constraintChecks": [],
        "solver": res["solver"],
        "riskAversionLambda": data.lam,
        "horizonMaturityLimits": data.horizon_maturity,
        "analysis": {},
    }
    if res["status"] not in OPTIMAL:
        return out

    pos = res["positions"]
    ev = evaluate(data, pos)
    out["metrics"] = _metrics_block(ev, data.alpha) | {"cvarLp": res["cvar_lp"]}
    out["weights"] = _weights(data, pos)
    out["constraintChecks"] = check_constraints(data, pos)

    ev0 = evaluate(data, data.x0)
    e_star = ev["expected_pnl"] - ev["transaction_cost"]
    mv = solve_data(data, mode="variance", min_net_return=e_star)
    out["analysis"] = {
        "initialPosition": {
            "objective": ev0["expected_pnl"] - data.lam * ev0["cvar"],
            "metrics": _metrics_block(ev0, data.alpha),
            "weights": _weights(data, data.x0),
            "violatedConstraints": [c["name"] for c in check_constraints(data, data.x0) if not c["ok"]],
        },
        "lambdaSweep": [_summary(data, solve_data(data, lam=lam), lam) for lam in data.lambda_grid],
        "meanVarianceBenchmark": {"minNetReturn": e_star, **_summary(data, mv, data.lam)},
        "horizonMaturityVariant": _horizon_variant(data, res),
    }
    return out


def out_of_sample(doc: Mapping, positions: pd.Series, seed: int) -> dict:
    """Evalúa una cartera con escenarios generados con otra semilla (§9.4): mide el sobreajuste del CVaR."""
    scen = risk.build_scenarios(doc, seed=seed)
    reval, flows = risk.scenario_components(doc, scen.deltas)
    return evaluate(prepare(doc, components=(reval, flows, scen)), positions)

### (v) Sensibilidades S1–S9

(v) Sensibilidades del Caso C (docs/formulacion.md §11; decisiones D-21, D-22).

Cada variante es una copia del documento de input.json resuelta con el mismo optimizador. Las grillas salen de
`caseParameters.sensitivities`; nada se codifica aquí. Montos en S/ mm, pesos y tasas en decimales.

- S1 `rate_shock_pnl`: shock instantáneo de tasas en t0 sobre carteras fijas (revalorización completa, spreads fijos).
- S2 `shadow_prices`: duales del LP de cada cota de peso activa, contrastados con una diferencia finita.
- S3-S9 `run_all`: costos, α, peso de la cola de estrés, semillas, deriva, estrés extra y regla de cupones.
  La matriz r_{k,s} se reutiliza siempre que la variante no cambie los escenarios ni la valorización.

In [ ]:
%%writefile {LIB}/optimum/sensitivity.py
"""(v) Sensibilidades del Caso C (docs/formulacion.md §11; decisiones D-21, D-22).

Cada variante es una copia del documento de input.json resuelta con el mismo optimizador. Las grillas salen de
`caseParameters.sensitivities`; nada se codifica aquí. Montos en S/ mm, pesos y tasas en decimales.

- S1 `rate_shock_pnl`: shock instantáneo de tasas en t0 sobre carteras fijas (revalorización completa, spreads fijos).
- S2 `shadow_prices`: duales del LP de cada cota de peso activa, contrastados con una diferencia finita.
- S3-S9 `run_all`: costos, α, peso de la cola de estrés, semillas, deriva, estrés extra y regla de cupones.
  La matriz r_{k,s} se reutiliza siempre que la variante no cambie los escenarios ni la valorización.
"""

from __future__ import annotations

import copy
import dataclasses
import time
from collections.abc import Mapping

import numpy as np
import pandas as pd

from optimum import optimizer as opt
from optimum import risk
from optimum.valuation import calibrate_spreads, coupon_rule, value_at_t0

PP = 100.0  # puntos porcentuales por unidad de peso
EXCLUDED_LIMITS = {"assetBudget", "liabilityBudget", "nonNegativity"}  # no son límites que se puedan relajar


# ------------------------------------------------------------------ copias del documento
def with_overrides(doc: Mapping, overrides: Mapping[str, object]) -> dict:
    """Copia profunda de `doc` con `{"a.b.c": valor}` reemplazados. KeyError si la ruta no existe (evita que un
    error de tipeo cree una clave nueva que nadie lee)."""
    new = copy.deepcopy(dict(doc))
    for path, value in overrides.items():
        *parents, last = path.split(".")
        node = new
        for key in parents:
            node = node[key]
        if last not in node:
            raise KeyError(f"La ruta {path} no existe en el documento")
        node[last] = copy.deepcopy(value)
    return new


def scale_costs(doc: Mapping, multiplier: float) -> dict:
    """Copia con los costos de transacción × `multiplier` (S3)."""
    if multiplier < 0:
        raise ValueError(f"El multiplicador de costos debe ser ≥ 0: {multiplier}")
    new = copy.deepcopy(dict(doc))
    for c in new["transactionCosts"].values():
        for key in ("restructuring_cost", "reduction_prepayment_cost"):
            if c.get(key) is not None:
                c[key] = float(c[key]) * multiplier
    return new


def add_stress(doc: Mapping, scenario: Mapping) -> dict:
    """Copia con un escenario de estrés más (S8): su probabilidad se suma a `stressProbability` y se resta de
    `bootstrapProbability`, para que Σ p_s = 1."""
    new = copy.deepcopy(dict(doc))
    sid = scenario["scenario_id"]
    if any(s["scenario_id"] == sid for s in new["stressScenarios"]):
        raise ValueError(f"Ya existe un estrés con id {sid}")
    p = float(scenario["probability"])
    cp_ = new["caseParameters"]
    if not 0 < p < float(cp_["bootstrapProbability"]):
        raise ValueError(f"Probabilidad del estrés {sid} fuera de rango: {p}")
    new["stressScenarios"].append(dict(scenario))
    cp_["stressProbability"] = float(cp_["stressProbability"]) + p
    cp_["bootstrapProbability"] = float(cp_["bootstrapProbability"]) - p
    return new


def reweight_stress(
    scen: risk.ScenarioSet, stress_probability: float | None = None, drop: str | None = None
) -> risk.ScenarioSet:
    """Nuevas probabilidades del conjunto de escenarios (S5), sin tocar los Δ.

    `drop` elimina un estrés. La probabilidad conjunta P (`stress_probability`, o la original) se reparte entre los
    estrés restantes en proporción a su p original; cada bootstrap recibe (1 − P)/N_B.
    """
    kind = np.asarray(scen.kind)
    keep = np.ones(len(scen.ids), dtype=bool)
    if drop is not None:
        if drop not in scen.ids or kind[scen.ids.index(drop)] != "stress":
            raise ValueError(f"{drop} no es un escenario de estrés del conjunto")
        keep[scen.ids.index(drop)] = False
    is_stress = (kind == "stress") & keep
    if not is_stress.any():
        raise ValueError("El conjunto quedaría sin escenarios de estrés")
    p = np.asarray(scen.probs, dtype=float)
    total = float(p[kind == "stress"].sum()) if stress_probability is None else float(stress_probability)
    if not 0 < total < 1:
        raise ValueError(f"La probabilidad conjunta del estrés debe estar en (0, 1): {total}")
    is_boot = (kind == "bootstrap") & keep
    new_p = np.zeros(len(p))
    new_p[is_stress] = total * p[is_stress] / p[is_stress].sum()
    new_p[is_boot] = (1 - total) / is_boot.sum()
    ids = [i for i, k in zip(scen.ids, keep, strict=True) if k]
    return risk.ScenarioSet(
        ids=ids,
        deltas=scen.deltas.loc[ids],
        probs=new_p[keep],
        kind=[k for k, m in zip(scen.kind, keep, strict=True) if m],
    )


# ------------------------------------------------------------------------ S1 · tasas
def rate_shock_delta(doc: Mapping, shock: Mapping) -> dict[str, float]:
    """Δ de los nodos spot para un shock de `rateShocks`: paralelo en `currencies`, o solo en el tramo `bucket` de
    `stressTenors` con la transición por t·s(t) de los estrés (D-11 v2). El resto de factores en 0."""
    node_years = {n["id"]: float(n["years"]) for n in doc["curveNodes"]}
    tenors = doc["caseParameters"]["stressTenors"]
    shift, bucket = float(shock["shift"]), shock.get("bucket")
    if bucket is not None and bucket not in tenors:
        raise ValueError(f"{shock['id']}: el tramo {bucket} no está en stressTenors ({sorted(tenors)})")
    delta = dict.fromkeys(risk.FACTORS, 0.0)
    for ccy in shock["currencies"]:
        if bucket is None:
            by_node = dict.fromkeys(node_years, shift)
        else:
            by_node = risk._stress_curve_shift(0.0, {bucket: shift}, tenors, node_years)
        for node, v in by_node.items():
            key = f"{ccy}_SPOT_{node}"
            if key not in delta:
                raise ValueError(f"{shock['id']}: factor desconocido {key}")
            delta[key] = float(v)
    return delta


def _with_rule(doc: Mapping, rule: str) -> dict:
    """Copia con `floatingCouponRule` = rule (la clave puede faltar: FLAT por defecto, D-22)."""
    new = copy.deepcopy(dict(doc))
    new["caseParameters"]["floatingCouponRule"] = rule
    return new


def rate_shock_pnl(
    doc: Mapping,
    portfolios: Mapping[str, pd.Series],
    spreads: Mapping[str, float] | None = None,
    rules: list[str] | None = None,
) -> pd.DataFrame:
    """ΔV instantáneo en t0 (S/ mm) de cada cartera bajo cada shock de `sensitivities.rateShocks` (S1).

    ΔV_k = x_k / V0_k · [V_k(t0; curva + δ) − V_k(t0)] por revalorización completa con los spreads calibrados
    fijos (flotantes reproyectados con la curva desplazada). Se repite para cada regla de cupones de `rules`
    (por defecto, solo la del documento): el tramo corto depende de D-13 (auditoría I-2). `spreads` solo se usa
    para la regla del documento; las demás se recalibran. Columnas: rule, shock, portfolio, assets (ΔV activos),
    liabilities (ΔV pasivos) y netWorth = assets − liabilities.
    """
    base_rule = coupon_rule(doc)
    inst = pd.DataFrame(doc["instruments"]).set_index("id")
    x0 = inst["market_value_pen"].astype(float)
    is_asset = inst["side"] == "ASSET"
    rows = []
    for rule in rules or [base_rule]:
        d = _with_rule(doc, rule)
        sp = spreads if (spreads is not None and rule == base_rule) else calibrate_spreads(d)
        v0 = value_at_t0(d, sp)
        for shock in doc["caseParameters"]["sensitivities"]["rateShocks"]:
            rel = ((value_at_t0(d, sp, rate_shock_delta(doc, shock)) - v0) / x0).reindex(inst.index)
            for name, pos in portfolios.items():
                change = pd.Series(pos, dtype=float).reindex(inst.index).fillna(0.0) * rel
                a, li = float(change[is_asset].sum()), float(change[~is_asset].sum())
                rows.append(
                    {
                        "rule": rule,
                        "shock": shock["id"],
                        "portfolio": name,
                        "assets": a,
                        "liabilities": li,
                        "netWorth": a - li,
                    }
                )
    return pd.DataFrame(rows)


# --------------------------------------------------------------------- variantes
def _metrics(data: opt.ProblemData, positions: pd.Series) -> dict:
    ev = opt.evaluate(data, positions)
    return {
        "expectedNetWorthChange": ev["expected_pnl"],
        "transactionCosts": ev["transaction_cost"],
        "expectedTerminalNetWorth": ev["expected_terminal_net_worth"],
        "varLoss": ev["var"],
        "cvarLoss": ev["cvar"],
        "cvarAlpha": data.alpha,
        "netWorthVolatility": ev["volatility"],
        "stressLosses": ev["stress_losses"],
        "tailStressShare": ev["tail_stress_share"],
    }


ROW_KEYS = (
    "objective",
    "expectedNetWorthChange",
    "transactionCosts",
    "expectedTerminalNetWorth",
    "varLoss",
    "cvarLoss",
    "cvarAlpha",
    "netWorthVolatility",
    "stressLosses",
    "tailStressShare",
    "turnover",
    "distanceToBase",
    "weights",
    "activeLimits",
    "positions",
)


def _row(data: opt.ProblemData, res: dict, base_positions: pd.Series | None) -> dict:
    """Fila de resultados de una solución; si no es óptima, solo el status (el resto en None)."""
    if res["status"] not in opt.OPTIMAL:
        return {"status": res["status"], **dict.fromkeys(ROW_KEYS)}
    pos = res["positions"]
    x0 = data.x0.loc[data.ids]
    dist = None if base_positions is None else float((pos - base_positions.reindex(data.ids)).abs().sum())
    return {
        "status": res["status"],
        "objective": res["objective"],
        **_metrics(data, pos),
        "turnover": float((pos - x0).abs().sum()),
        "distanceToBase": dist,
        "weights": opt._weights(data, pos),
        "activeLimits": [
            c["name"]
            for c in opt.check_constraints(data, pos)
            if c["active"] and c["name"] not in EXCLUDED_LIMITS
        ],
        "positions": {k: float(v) for k, v in pos.items()},
    }


def solve_variant(doc: Mapping, components: tuple, base_positions: pd.Series | None = None) -> dict:
    """Resuelve una variante con λ = riskAversionLambda y devuelve su fila (ver `_row`). `components` =
    (reval, flows, ScenarioSet); si el ScenarioSet es un subconjunto, se usan solo sus columnas. Con
    `base_positions`, la fila trae además `basePortfolio`: esa cartera evaluada con el modelo de la variante
    (separa el efecto de la medida del de reoptimizar, auditoría M-2)."""
    data = opt.prepare(doc, components=components)
    row = _row(data, opt.solve_data(data), base_positions)
    if base_positions is not None:
        row["basePortfolio"] = _metrics(data, base_positions)
    return row


# ------------------------------------------------------------------ S2 · precios sombra
def _active_bound(check: dict) -> str | None:
    """'min' o 'max' según la cota en la que está un límite activo."""
    v = check["value"]
    if check["max"] is not None and abs(v - check["max"]) <= opt.ACTIVE_TOL * max(1.0, abs(check["max"])):
        return "max"
    if check["min"] is not None and abs(v - check["min"]) <= opt.ACTIVE_TOL * max(1.0, abs(check["min"])):
        return "min"
    return None


def _twins(data: opt.ProblemData, active: Mapping[str, str]) -> list[tuple[str, str]]:
    """Parejas de cotas activas gemelas (auditoría I-3): mismo lado y dimensión, miembros disjuntos que cubren todo
    el lado, una en su mínimo y otra en su máximo con min + max = 1 (p. ej. pasivo fijo ≥ 45 % y flotante ≤ 55 %).
    Son la misma restricción: el LP reparte el dual entre ambas de forma arbitraria. Devuelve (nombre en mínimo,
    nombre en máximo)."""
    groups = {g["name"]: g for g in data.groups}
    pairs = []
    for lo_name, lo_bound in active.items():
        if lo_bound != "min":
            continue
        lo = groups[lo_name]
        side_ids = {i for i in data.ids if data.side[i] == lo["side"]}
        for hi_name, hi_bound in active.items():
            hi = groups[hi_name]
            if (
                hi_bound == "max"
                and hi["side"] == lo["side"]
                and hi["dimension"] == lo["dimension"]
                and not set(lo["members"]) & set(hi["members"])
                and set(lo["members"]) | set(hi["members"]) == side_ids
                and abs(lo["min"] + hi["max"] - 1.0) <= opt.OK_TOL
            ):
                pairs.append((lo_name, hi_name))
    return pairs


def _relaxed_gain(data: opt.ProblemData, base_obj: float, moves: Mapping[str, str], relax: float) -> tuple:
    """(ganancia por pp, pp relajados) al relajar a la vez las cotas `moves` {grupo: 'min'|'max'} en `relax`
    (acotado a [0, 1]; el mismo paso para todas). (None, 0) si no se puede relajar."""
    groups = {g["name"]: g for g in data.groups}
    room = [groups[n]["min"] if b == "min" else 1.0 - groups[n]["max"] for n, b in moves.items()]
    step = min([relax, *room])
    if step <= 0:
        return None, 0.0
    sign = {"min": -1.0, "max": 1.0}
    relaxed = dataclasses.replace(
        data,
        groups=[
            {**g, moves[g["name"]]: g[moves[g["name"]]] + sign[moves[g["name"]]] * step}
            if g["name"] in moves
            else g
            for g in data.groups
        ],
    )
    res = opt.solve_data(relaxed)
    if res["status"] not in opt.OPTIMAL:
        return None, step * PP
    return (res["objective"] - base_obj) / (step * PP), step * PP


def shadow_prices(doc: Mapping, components: tuple) -> pd.DataFrame:
    """Precio sombra de cada cota de peso activa del Cuadro 4 (S2): S/ mm de objetivo por 1 pp de relajación.

    - dualPerPp = dual del LP / 100: derivada del objetivo al relajar la cota con las demás fijas.
    - finiteDiffPerPp / finiteDiffSmallPerPp = (objetivo relajado − objetivo) / pp, relajando `limitRelaxation` y
      `limitRelaxationSmall` (subir el máximo o bajar el mínimo, dentro de [0, 1]). El objetivo óptimo es cóncavo y
      lineal por tramos en el límite: el dual es la pendiente local (≈ paso chico) y el paso grande es la ganancia
      media del tramo, menor si la pendiente cae (auditoría M-1).
    - Cotas gemelas (`_twins`, auditoría I-3) van en una sola fila "A+B" que las relaja juntas; su dual es la suma.
    Las cotas que no se pueden relajar (mínimo 0, máximo 1) se omiten.
    """
    cfg = doc["caseParameters"]["sensitivities"]
    relax, small = float(cfg["limitRelaxation"]), float(cfg["limitRelaxationSmall"])
    data = opt.prepare(doc, components=components)
    base = opt.solve_data(data, duals=True)
    if base["status"] not in opt.OPTIMAL:
        raise ValueError(f"El problema base no es óptimo ({base['status']}): no hay precios sombra")
    groups = {g["name"]: g for g in data.groups}
    checks = {c["name"]: c for c in opt.check_constraints(data, base["positions"])}
    active = {
        n: b for n, c in checks.items() if n in groups and c["active"] and (b := _active_bound(c)) is not None
    }
    twins = _twins(data, active)
    paired = {n for pair in twins for n in pair}
    entries = [({lo: "min", hi: "max"}, f"{lo}+{hi}") for lo, hi in twins]
    entries += [({n: b}, n) for n, b in active.items() if n not in paired]

    rows = []
    for moves, name in entries:
        fd, pp = _relaxed_gain(data, base["objective"], moves, relax)
        if pp == 0:
            continue
        fd_small, pp_small = _relaxed_gain(data, base["objective"], moves, small)
        first = next(iter(moves))
        rows.append(
            {
                "name": name,
                "side": groups[first]["side"],
                "bound": "+".join(moves.values()),
                "limit": float(groups[first][moves[first]]),
                "value": checks[first]["value"],
                "dualPerPp": sum(base["duals"][f"{n}:{b}"] for n, b in moves.items()) / PP,
                "finiteDiffPerPp": fd,
                "finiteDiffSmallPerPp": fd_small,
                "relaxationPp": pp,
                "relaxationSmallPp": pp_small,
            }
        )
    cols = [
        "side",
        "bound",
        "limit",
        "value",
        "dualPerPp",
        "finiteDiffPerPp",
        "finiteDiffSmallPerPp",
        "relaxationPp",
        "relaxationSmallPp",
    ]
    return pd.DataFrame(rows, columns=["name", *cols]).set_index("name")


# ------------------------------------------------------------------ corrida completa
def _components(doc: Mapping, seed: int | None = None) -> tuple:
    scen = risk.build_scenarios(doc, seed=seed)
    reval, flows = risk.scenario_components(doc, scen.deltas)
    return reval, flows, scen


def extra_stress_components(doc: Mapping, components: tuple, scenario: Mapping) -> tuple[dict, tuple]:
    """(doc con el estrés agregado, componentes) para S8 sin revalorizar todo: solo se revaloriza el escenario
    nuevo y se concatena a r_{k,s} base. Válido porque el bootstrap no cambia (misma semilla y bootstrapCount:
    mismos ids y Δ) y `prepare` selecciona columnas por id."""
    reval, flows, _ = components
    new_doc = add_stress(doc, scenario)
    scen = risk.build_scenarios(new_doc)
    sid = scenario["scenario_id"]
    r_new, f_new = risk.scenario_components(new_doc, scen.deltas.loc[[sid]])
    return new_doc, (pd.concat([reval, r_new], axis=1), pd.concat([flows, f_new], axis=1), scen)


def _records(df: pd.DataFrame) -> list[dict]:
    return [
        {k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in r.items()}
        for r in df.to_dict("records")
    ]


def run_all(doc: Mapping, log=print) -> dict:
    """Todas las sensibilidades de `caseParameters.sensitivities` (S1-S9), serializable a JSON.

    Cada fila de reoptimización trae la distancia L1 a la cartera óptima base y esa cartera evaluada con el modelo
    de la variante (`basePortfolio`; en S6 se llama `outOfSample`). S1 se reporta con la regla de cupones del
    documento y con cada una de `floatingCouponRules`. `log` recibe mensajes de avance (None = silencio).
    """
    log = log or (lambda *_: None)
    cfg = doc["caseParameters"]["sensitivities"]
    start = time.perf_counter()

    log("base: escenarios y revalorización…")
    comps = _components(doc)
    reval, flows, scen = comps
    data = opt.prepare(doc, components=comps)
    res = opt.solve_data(data)
    base = _row(data, res, None)
    if res["status"] not in opt.OPTIMAL:
        return {"base": base, "error": "El problema base no es óptimo: no se corren sensibilidades"}
    x_star = res["positions"]
    e_star = base["expectedNetWorthChange"] - base["transactionCosts"]
    mv = opt.solve_data(data, mode="variance", min_net_return=e_star)
    portfolios = {"initial": data.x0.loc[data.ids], "optimal": x_star}
    if mv["status"] in opt.OPTIMAL:
        portfolios["meanVariance"] = mv["positions"]

    out: dict = {"base": base}
    log("S1 tasas, S2 precios sombra…")
    rules = [coupon_rule(doc), *(r for r in cfg["floatingCouponRules"] if r != coupon_rule(doc))]
    out["rateShocks"] = _records(rate_shock_pnl(doc, portfolios, rules=rules))
    out["shadowPrices"] = _records(shadow_prices(doc, comps).reset_index())

    log("S3 costos, S4 α, S5 cola de estrés…")
    out["costs"] = [
        {"multiplier": float(m), **solve_variant(scale_costs(doc, m), comps, x_star)}
        for m in cfg["costMultipliers"]
    ]
    out["alphas"] = [
        {
            "alpha": float(a),
            **solve_variant(with_overrides(doc, {"caseParameters.cvarAlpha": a}), comps, x_star),
        }
        for a in cfg["cvarAlphas"]
    ]
    out["stressWeight"] = [
        {
            "stressProbability": float(p),
            **solve_variant(doc, (reval, flows, reweight_stress(scen, stress_probability=p)), x_star),
        }
        for p in cfg["stressProbabilities"]
    ]
    stress_ids = [s for s, k in zip(scen.ids, scen.kind, strict=True) if k == "stress"]
    out["stressLeaveOneOut"] = (
        [
            {"dropped": s, **solve_variant(doc, (reval, flows, reweight_stress(scen, drop=s)), x_star)}
            for s in stress_ids
        ]
        if cfg.get("stressLeaveOneOut")
        else []
    )

    out["seeds"] = []
    for seed in cfg["seeds"]:
        log(f"S6 semilla {seed}…")
        row = solve_variant(doc, _components(doc, seed=int(seed)), x_star)
        out["seeds"].append({"seed": int(seed), "outOfSample": row.pop("basePortfolio"), **row})

    out["drift"] = []
    for name, spec in cfg["driftVariants"].items():
        log(f"S7 deriva {name}…")
        dd = with_overrides(doc, {"caseParameters.drift": spec})
        out["drift"].append({"variant": name, **solve_variant(dd, _components(dd), x_star)})

    out["extraStress"] = []
    for extra in cfg["extraStress"]:
        log(f"S8 estrés {extra['scenario_id']}…")
        dd, c = extra_stress_components(doc, comps, extra)
        out["extraStress"].append({"scenario": extra["scenario_id"], **solve_variant(dd, c, x_star)})

    out["floatingCouponRule"] = []
    for rule in cfg["floatingCouponRules"]:
        log(f"S9 regla de cupones {rule}…")
        dd = _with_rule(doc, rule)
        out["floatingCouponRule"].append({"rule": rule, **solve_variant(dd, _components(dd), x_star)})

    out["meta"] = {
        "caseId": doc["caseId"],
        "riskAversionLambda": data.lam,
        "baseSeed": doc["caseParameters"]["seed"],
        "config": copy.deepcopy(dict(cfg)),
        "runtimeSeconds": time.perf_counter() - start,
    }
    return out

### Etiquetas en español

Etiquetas en español para tablas y figuras del informe (los identificadores del JSON quedan en inglés).

In [ ]:
%%writefile {LIB}/optimum/reporting/labels.py
"""Etiquetas en español para tablas y figuras del informe (los identificadores del JSON quedan en inglés)."""

from __future__ import annotations

SIDES = {"ASSET": "Activos", "LIABILITY": "Pasivos"}
DIMENSIONS = {
    "CURRENCY": "Moneda",
    "TYPE": "Tipo",
    "RATE_TYPE": "Tasa",
    "MATURITY": "Vencimiento",
    "CONCENTRATION": "Concentración",
}
CATEGORIES = {
    "CASH": "caja",
    "FIXED_INCOME": "renta fija",
    "EQUITY": "renta variable",
    "FIXED": "fija",
    "FLOAT": "flotante",
    "LE_12M": "≤ 12 meses",
    "GT_36M": "> 36 meses",
    "SINGLE_POSITION": "",
}
SCALARS = {
    "assetBudget": "Presupuesto de activos",
    "liabilityBudget": "Presupuesto de pasivos",
    "minCash": "Caja mínima",
    "minNetWorth": "PN mínimo",
    "maxLiabilitiesToAssets": "P/A máximo",
    "nonNegativity": "No negatividad",
}
COUPON_RULES = {"FLAT": "nivel plano", "PERIOD_FORWARD": "forward del período"}
HORIZON_MODES = {"ENFORCE": "vencimientos exigidos al horizonte", "REPORT": "vencimientos solo reportados"}


def constraint_label(name: str) -> str:
    """'ASSET:TYPE:FIXED_INCOME' → 'Activos · Tipo · renta fija'; 'LIABILITY:CONCENTRATION:SINGLE_POSITION:L01' →
    'Pasivos · Concentración · L01'; escalares ('minCash') → nombre en español. Categorías desconocidas quedan igual."""
    if name in SCALARS:
        return SCALARS[name]
    parts = name.split(":")
    if len(parts) < 2:
        return name
    out = [SIDES.get(parts[0], parts[0]), DIMENSIONS.get(parts[1], parts[1])]
    out += [CATEGORIES.get(p, p) for p in parts[2:]]
    return " · ".join(p for p in out if p)


def coupon_rule(rule: str) -> str:
    return COUPON_RULES.get(rule, rule)


def horizon_mode(mode: str) -> str:
    return HORIZON_MODES.get(mode, mode)

### Importación y estilo de gráficos

In [ ]:
import json
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import colormaps

for _m in [m for m in sys.modules if m == "optimum" or m.startswith("optimum.")]:
    del sys.modules[_m]  # si se re-ejecuta, recargar el código recién escrito
import optimum

assert Path(optimum.__file__).is_relative_to(LIB), f"se importó otro paquete optimum: {optimum.__file__}"

from optimum import optimizer as opt
from optimum import risk, sensitivity
from optimum.io.json_contract import _clean, validate_input, validate_output
from optimum.reporting.labels import SIDES, constraint_label, coupon_rule, horizon_mode
from optimum.valuation import calibrate_spreads, value_at_t0

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

COLORS = {
    "PEN": "#1f4e79",
    "USD": "#c55a11",
    "ASSET": "#2e7d32",
    "LIABILITY": "#b71c1c",
    "EQUITY": "#6a1b9a",
    "neutral": "#595959",
}
mpl.rcParams.update(
    {
        "figure.figsize": (9, 4.5),
        "figure.dpi": 100,
        "font.size": 9,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)
FIG_DIR = Path("figuras")


def show(fig, name: str) -> None:
    # Muestra la figura y, si SAVE_FIGURES, la guarda en figuras/<name>.png
    fig.tight_layout()
    if SAVE_FIGURES:
        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 3. Input

Se lee y valida `input.json` contra el contrato v1.0.

In [ ]:
path = Path(INPUT_PATH).expanduser()
while not path.is_file():
    answer = input(f"No se encontró '{path}'. Ruta del input.json: ").strip().strip('"').strip("'")
    if not answer:
        raise FileNotFoundError("No se indicó un input.json")
    path = Path(answer).expanduser()

doc = json.loads(path.read_text(encoding="utf-8"))
errors = validate_input(doc)
if errors:
    raise ValueError("input.json inválido:\n- " + "\n- ".join(errors))

cp_ = doc["caseParameters"]
com = doc["constraints"]["common"]
lam, alpha = cp_["riskAversionLambda"], cp_["cvarAlpha"]
tag = f"{alpha * 100:g}"
inst = pd.DataFrame(doc["instruments"]).set_index("id")
hist = risk.history_from_doc(doc)
print(f"{path.resolve()}")
print(f"Caso {doc['caseId']} · valorización {doc['valuationDate']} → horizonte {doc['horizonDate']} · "
      f"moneda base {doc['baseCurrency']}")
print(f"{len(doc['marketHistory'])} observaciones de mercado ({hist.index[0]:%Y-%m} a {hist.index[-1]:%Y-%m}) · "
      f"{len(inst)} instrumentos · {len(doc['cashFlows'])} flujos · {len(doc.get('stressScenarios', []))} estrés")
print(f"λ = {lam} · α = {alpha} · medida {cp_.get('riskMeasure')} · {cp_['bootstrapCount']} escenarios bootstrap "
      f"(p = {cp_['bootstrapProbability']}) · semilla {cp_['seed']}")

## 4. Datos de entrada

### Balance inicial

In [ ]:
bal = inst[["side", "name", "instrument_type", "currency", "market_value_pen", "notional_native", "maturity"]].copy()
bal["side"] = bal["side"].map(SIDES)
pn0 = inst.loc[inst["side"] == "ASSET", "market_value_pen"].sum() - inst.loc[inst["side"] == "LIABILITY", "market_value_pen"].sum()
print(f"Activos {inst.loc[inst['side'] == 'ASSET', 'market_value_pen'].sum():,.1f} · "
      f"Pasivos {inst.loc[inst['side'] == 'LIABILITY', 'market_value_pen'].sum():,.1f} · PN0 {pn0:,.1f} S/ mm")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, side, title in zip(axes, ["ASSET", "LIABILITY"], ["Activos", "Pasivos"], strict=True):
    t = inst[inst["side"] == side]
    ax.bar(t.index, t["market_value_pen"], color=[COLORS[c] for c in t["currency"]])
    for k, (_, r) in enumerate(t.iterrows()):
        ax.annotate(r["instrument_type"].replace("BOND_", ""), (k, r["market_value_pen"]), ha="center",
                    va="bottom", fontsize=7)
    ax.set_title(f"{title} ({t['market_value_pen'].sum():,.0f} S/ mm)")
axes[0].set_ylabel("Valor de mercado (S/ mm)")
axes[1].legend(handles=[mpl.patches.Patch(color=COLORS[c], label=c) for c in ["PEN", "USD"]])
fig.suptitle("Balance inicial por instrumento (color = moneda)")
show(fig, "balance_inicial")
bal

### Curvas de tasas

In [ ]:
t0 = hist.index[-1]
nodes = [n["id"] for n in doc["curveNodes"]]
years = [n["years"] for n in doc["curveNodes"]]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ccy in ["PEN", "USD"]:
    axes[0].plot(years, 100 * hist.loc[t0, [f"{ccy}_SPOT_{n}" for n in nodes]], "-o", ms=4, color=COLORS[ccy],
                 label=f"{ccy} {t0:%Y-%m-%d}")
    axes[0].plot(years, 100 * hist.iloc[0][[f"{ccy}_SPOT_{n}" for n in nodes]], ":", color=COLORS[ccy],
                 label=f"{ccy} {hist.index[0]:%Y-%m-%d}")
axes[0].set(title="Curvas spot (inicio y t0)", xlabel="Plazo (años)", ylabel="Tasa spot (%)")
axes[0].legend(fontsize=7)
for ax, ccy in zip(axes[1:], ["PEN", "USD"], strict=True):
    for n, shade in zip(nodes, np.linspace(0.35, 1.0, len(nodes)), strict=True):
        ax.plot(hist.index, 100 * hist[f"{ccy}_SPOT_{n}"], color=COLORS[ccy], alpha=shade, lw=1, label=n)
    ax.set(title=f"Historia spot {ccy}", ylabel="%")
    ax.legend(fontsize=6, ncol=3)
for ax in axes[1:]:
    ax.tick_params(axis="x", labelrotation=30)
show(fig, "curvas_spot")

### Tipo de cambio y renta variable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(hist.index, hist["FX_PENUSD"], color=COLORS["USD"])
axes[0].set(title="Tipo de cambio PEN por USD", ylabel="PEN/USD")
axes[1].plot(hist.index, hist["EQUITY_USD"], color=COLORS["EQUITY"])
axes[1].set(title="Índice de renta variable (USD)", ylabel="Nivel")
fig.autofmt_xdate()
show(fig, "fx_equity")

### Perfil de flujos de caja

Flujos de `cashFlows` (los flotantes son la proyección base; en cada escenario se reproyectan).

In [ ]:
cf = pd.DataFrame(doc["cashFlows"])
cf["year"] = pd.to_datetime(cf["payment_date"]).dt.year
fx0 = hist.loc[t0, "FX_PENUSD"]
cf["pen"] = cf["total_cash_flow"] * np.where(cf["currency"] == "USD", fx0, 1.0)  # al tipo de cambio de t0
prof = cf.pivot_table(index="year", columns="side", values="pen", aggfunc="sum").fillna(0.0)
fig, ax = plt.subplots(figsize=(9, 3.5))
k = np.arange(len(prof))
ax.bar(k - 0.2, prof.get("ASSET", 0), 0.4, color=COLORS["ASSET"], label="Activos (cobros)")
ax.bar(k + 0.2, -prof.get("LIABILITY", 0), 0.4, color=COLORS["LIABILITY"], label="Pasivos (pagos)")
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(k, prof.index)
ax.set(title="Flujos contractuales/proyectados por año (S/ mm, FX de t0)", ylabel="S/ mm")
ax.legend()
show(fig, "perfil_flujos")

### Factores de riesgo

Cambios mensuales: Δ absoluta en tasas, log-retorno en FX y equity. La covarianza es singular porque las forwards se derivan de las spots.

In [ ]:
chg = risk.factor_changes(doc)
corr = chg.corr()
cov = risk.covariance(chg)
eig = np.sort(np.linalg.eigvalsh(cov.to_numpy()))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.3, 1]})
im = axes[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
axes[0].set_yticks(range(len(corr)), corr.index, fontsize=6)
axes[0].grid(False)
fig.colorbar(im, ax=axes[0], shrink=0.8)
axes[0].set_title("Correlación de cambios mensuales de factores")
axes[1].semilogy(np.arange(1, len(eig) + 1), np.clip(eig, 1e-20, None), "o-", ms=3, color=COLORS["neutral"])
axes[1].set(title="Autovalores de Σ anualizada (escala log)", xlabel="Componente", ylabel="Autovalor")
show(fig, "correlacion_factores")
var_share = np.cumsum(eig) / eig.sum()
print(f"{len(chg)} cambios mensuales · 3 primeros componentes explican {100 * var_share[2]:.1f} % de la varianza · "
      f"autovalores ≈ 0 (< 1e-10 × máx.): {(eig < 1e-10 * eig.max()).sum()} (Σ singular: forwards derivadas de spots)")

### Límites y escenarios de estrés

In [ ]:
lim = pd.DataFrame(doc["constraints"]["weights"])
lim["Restricción"] = [constraint_label(f"{r.side}:{r.dimension}:{r.category}") for r in lim.itertuples()]
print("Restricciones comunes:", com)
display(lim.set_index("Restricción")[["min_weight", "max_weight", "note"]])
stress_in = pd.DataFrame(doc.get("stressScenarios", []))
stress_in

## 5. Modelo de riesgo

### Escenarios a un año y revalorización completa

Bootstrap por bloques de los cambios históricos más los escenarios de estrés; cada instrumento se revaloriza flujo a flujo en cada escenario (sin duración ni DV01).

In [ ]:
t_start = time.perf_counter()
scen = risk.build_scenarios(doc)
reval, flows = risk.scenario_components(doc, scen.deltas)  # revalorización completa por escenario
comps = (reval, flows, scen)
data = opt.prepare(doc, components=comps)
kind = np.asarray(scen.kind)
print(f"{len(scen.ids)} escenarios ({(kind == 'bootstrap').sum()} bootstrap + {(kind == 'stress').sum()} estrés), "
      f"Σp = {scen.probs.sum():.6f} · {time.perf_counter() - t_start:.0f} s")
mu = risk.drift(doc, chg)
mu[mu != 0].rename("deriva anual μ").to_frame()

### Calibración de spreads

El spread de cada instrumento se calibra para que el valor del modelo en $t_0$ iguale el valor de mercado.

In [ ]:
spreads = calibrate_spreads(doc)
v0 = value_at_t0(doc, spreads)
cal = pd.DataFrame(
    {
        "Mercado (S/ mm)": inst["market_value_pen"],
        "V0 modelo (S/ mm)": v0.reindex(inst.index),
        "Spread calibrado (pb)": 1e4 * pd.Series(spreads).reindex(inst.index),
        "Spread de referencia (pb)": 1e4 * pd.Series(doc["calibrationSpreads"]).reindex(inst.index),
    }
)
cal["Error (S/ mm)"] = cal["V0 modelo (S/ mm)"] - cal["Mercado (S/ mm)"]
print(f"Máximo error de calibración: {cal['Error (S/ mm)'].abs().max():.2e} S/ mm")
cal.round(4)

### Shocks y resultado por instrumento

In [ ]:
boot = scen.deltas[kind == "bootstrap"]
stress = scen.deltas[kind == "stress"]
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5))
panels = [
    ("PEN_SPOT_ON", "PEN ON (pb)", 1e4, COLORS["PEN"]),
    ("PEN_SPOT_10Y", "PEN 10Y (pb)", 1e4, COLORS["PEN"]),
    ("USD_SPOT_ON", "USD ON (pb)", 1e4, COLORS["USD"]),
    ("USD_SPOT_10Y", "USD 10Y (pb)", 1e4, COLORS["USD"]),
    ("FX_PENUSD", "FX (log-ret., %)", 100, COLORS["USD"]),
    ("EQUITY_USD", "Equity (log-ret., %)", 100, COLORS["EQUITY"]),
]
for ax, (f, label, k, color) in zip(axes.flat, panels, strict=True):
    ax.hist(k * boot[f], bins=30, color=color, alpha=0.7)
    for v in k * stress[f]:
        ax.axvline(v, color="black", ls="--", lw=0.8)
    ax.set_title(label)
fig.suptitle("Shocks a un año: bootstrap (histograma) y estrés (líneas)")
show(fig, "shocks_escenarios")

R = data.returns
p = data.probs
er = pd.DataFrame(
    {
        "lado": inst["side"].map(SIDES),
        "moneda": inst["currency"],
        "E[r] (%)": 100 * (R @ p),
        "σ(r) (%)": 100 * np.sqrt(((R.sub(R @ p, axis=0)) ** 2) @ p),
        "r estrés mín. (%)": 100 * R.loc[:, kind == "stress"].min(axis=1),
        "r estrés máx. (%)": 100 * R.loc[:, kind == "stress"].max(axis=1),
    }
)
fig, ax = plt.subplots(figsize=(8, 4))
for i, r in er.iterrows():
    ax.scatter(r["σ(r) (%)"], r["E[r] (%)"], color=COLORS[r["moneda"]],
               marker="o" if r["lado"] == SIDES["ASSET"] else "s", s=40)
    ax.annotate(i, (r["σ(r) (%)"], r["E[r] (%)"]), textcoords="offset points", xytext=(4, 3), fontsize=7)
ax.set(xlabel="σ(r) a un año (%)", ylabel="E[r] a un año (%)",
       title="Resultado por S/ invertido (● activos, ■ pasivos = costo; color = moneda)")
show(fig, "riesgo_retorno_instrumentos")
er.round(2)

## 6. Optimización → `output.json`

In [ ]:
t_start = time.perf_counter()
out = opt.solve(doc, components=comps)  # óptimo + posición inicial + barrido de λ + media-varianza + variante
errors = validate_output(out)
if errors:
    raise ValueError("output inválido:\n- " + "\n- ".join(errors))
Path(OUTPUT_PATH).write_text(json.dumps(_clean(out), indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{Path(OUTPUT_PATH).resolve()} · status {out['status']} · objetivo {out['objectiveValue']} · "
      f"{time.perf_counter() - t_start:.0f} s")
if out["status"] not in opt.OPTIMAL:
    raise RuntimeError(f"El problema no tiene solución óptima (status {out['status']}); output.json sin posiciones.")
if out["status"] == "OPTIMAL_INACCURATE":
    print("ADVERTENCIA: solución inexacta; revisar constraintChecks.")
an = out["analysis"]
ini, mv, hv = an["initialPosition"], an["meanVarianceBenchmark"], an["horizonMaturityVariant"]
label_opt, label_mv, label_hv = f"Óptimo (λ = {lam:g})", "Mínima varianza", "Venc. exigidos al horizonte"

## 7. Resultados

### Posiciones

In [ ]:
pos = pd.DataFrame(out["positions"]["assets"] + out["positions"]["liabilities"]).set_index("id")
pos_tab = pd.DataFrame(
    {
        "Lado": inst["side"].map(SIDES),
        "Instrumento": pos["name"],
        "Moneda": pos["currency"],
        "Inicial": pos["initial"],
        label_opt: pos["optimal"],
        "Cambio": pos["change"],
        label_mv: pd.Series(mv.get("positions", {})),
        label_hv: pd.Series(hv.get("positions", {})),
    }
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, side in zip(axes, ["ASSET", "LIABILITY"], strict=True):
    t = pos_tab[pos_tab["Lado"] == SIDES[side]]
    k = np.arange(len(t))
    ax.bar(k - 0.2, t["Inicial"], 0.38, color=COLORS["neutral"], alpha=0.45, label="Inicial")
    ax.bar(k + 0.2, t[label_opt], 0.38, color=COLORS[side], label=label_opt)
    ax.scatter(k + 0.2, t[label_hv], marker="_", s=180, color="black", zorder=3, label=label_hv)
    ax.set_xticks(k, [f"{i}\n{c}" for i, c in zip(t.index, t["Moneda"], strict=True)], fontsize=7)
    ax.set_title(SIDES[side])
axes[0].set_ylabel("S/ mm")
axes[1].legend(fontsize=7)
fig.suptitle("Posiciones inicial y óptima")
show(fig, "posiciones_inicial_optima")
pos_tab.round(2)

### Métricas: inicial, óptimo y alternativas

Pérdida > 0 significa que el patrimonio cae.

In [ ]:
def metric_rows(m: dict, objective: float) -> dict:
    return {
        "Objetivo (E − λ·CVaR − TC)": objective,
        "E[ΔPN]": m["expectedNetWorthChange"],
        "Costo de transacción": m["transactionCosts"],
        "E[ΔPN] − TC": m["expectedNetWorthChange"] - m["transactionCosts"],
        "E[PN] al horizonte": m["expectedTerminalNetWorth"],
        "σ(ΔPN)": m["netWorthVolatility"],
        f"VaR {tag} %": m[f"var{tag}Loss"],
        f"CVaR {tag} %": m[f"cvar{tag}Loss"],
        "Rentabilidad esperada activos (%)": 100 * m["expectedAssetReturn"],
        "Costo esperado pasivos (%)": 100 * m["expectedLiabilityEconomicCost"],
        "Estrés en la cola (%)": 100 * m["tailStressShare"],
        **{f"Pérdida {k}": v for k, v in m["stressLosses"].items()},
    }


cols = {"Inicial": metric_rows(ini["metrics"], ini["objective"]),
        label_opt: metric_rows(out["metrics"], out["objectiveValue"])}
for name, alt in [(label_mv, mv), (label_hv, hv)]:
    if alt.get("metrics"):
        cols[name] = metric_rows(alt["metrics"], alt["objective"])
met = pd.DataFrame(cols)
print("Límites que incumple la posición inicial:", [constraint_label(c) for c in ini["violatedConstraints"]] or "ninguno")
met.round(2)

### Distribución de ΔPN y estrés

In [ ]:
positions = {"Inicial": data.x0, label_opt: pos["optimal"]}
if mv.get("positions"):
    positions[label_mv] = pd.Series(mv["positions"])
ev = {k: opt.evaluate(data, v) for k, v in positions.items()}
assert abs(ev[label_opt]["cvar"] - out["metrics"][f"cvar{tag}Loss"]) < 1e-6  # coherencia con output.json

r = data.returns.to_numpy()
styles = {"Inicial": COLORS["neutral"], label_opt: COLORS["ASSET"], label_mv: COLORS["USD"]}
dpns = {name: (data.sign * x.reindex(data.ids).to_numpy()) @ r for name, x in positions.items()}
bins = np.linspace(np.floor(min(d.min() for d in dpns.values())), np.ceil(max(d.max() for d in dpns.values())), 61)
fig, ax = plt.subplots(figsize=(10, 4.2))
for j, (name, dpn) in enumerate(dpns.items()):
    ax.hist(dpn[kind == "bootstrap"], bins=bins, histtype="step", lw=1.4, color=styles[name], label=name)
    ax.scatter(dpn[kind == "stress"], np.full((kind == "stress").sum(), -2.0 - 3 * j), marker="v", s=28,
               color=styles[name], zorder=3)
    ax.axvline(-ev[name]["cvar"], color=styles[name], ls=":", lw=1.2)
ax.set(xlabel="ΔPN a un año (S/ mm)", ylabel="Escenarios bootstrap",
       title=f"Distribución de ΔPN (▼ estrés; punteada: −CVaR {tag} %)")
ax.legend(fontsize=8)
show(fig, "distribucion_dpn")

sl = pd.DataFrame({"Inicial": ini["metrics"]["stressLosses"], label_opt: out["metrics"]["stressLosses"]})
fig, ax = plt.subplots(figsize=(8, 3.2))
k = np.arange(len(sl))
ax.barh(k + 0.2, sl["Inicial"], 0.4, color=COLORS["neutral"], alpha=0.6, label="Inicial")
ax.barh(k - 0.2, sl[label_opt], 0.4, color=COLORS["ASSET"], label=label_opt)
ax.axvline(0, color="black", lw=0.6)
ax.set_yticks(k, sl.index)
ax.set(xlabel="Pérdida (S/ mm; < 0 = ganancia)", title="Pérdida en cada escenario de estrés")
ax.legend(fontsize=8)
show(fig, "perdidas_estres")

### Restricciones en el óptimo

Pesos como fracción del total del lado; *activa* = en su cota.

In [ ]:
chk = pd.DataFrame(out["constraintChecks"])
chk_tab = pd.DataFrame(
    {"Valor": chk["value"], "Mín.": chk["min"], "Máx.": chk["max"], "Cumple": np.where(chk["ok"], "sí", "NO"),
     "Activa": np.where(chk["active"], "sí", ""), "Nota": chk["note"]}
)
chk_tab.index = chk["name"].map(constraint_label)
print(f"{chk['active'].sum()} restricciones activas de {len(chk)} · incumplidas: {(~chk['ok']).sum()}")

dims = chk["name"].str.split(":")
wt = chk[dims.str.len().ge(3) & dims.str[1].isin(["CURRENCY", "TYPE", "RATE_TYPE", "MATURITY"])]
fig, ax = plt.subplots(figsize=(9, 0.32 * len(wt) + 1))
y = np.arange(len(wt))[::-1]
lo = wt["min"].astype(float).fillna(0.0).to_numpy()
hi = wt["max"].astype(float).fillna(1.0).to_numpy()
ax.barh(y, 100 * (hi - lo), left=100 * lo, color=COLORS["neutral"], alpha=0.2, label="Rango permitido")
ax.scatter(100 * wt["value"], y, color=np.where(wt["active"], COLORS["LIABILITY"], COLORS["ASSET"]), zorder=3,
           label="Peso óptimo (rojo = activa)")
ax.set_yticks(y, wt["name"].map(constraint_label), fontsize=7)
ax.set(xlabel="% del lado", title="Pesos del óptimo frente a los límites")
ax.legend(fontsize=7, loc="lower right")
show(fig, "pesos_limites")
chk_tab

### Barrido de aversión al riesgo (λ)

In [ ]:
def w(s, dim, side, cat):
    return 100 * s["weights"][dim][side].get(cat, 0.0) if s.get("weights") else np.nan


sweep = pd.DataFrame(
    [
        {
            "λ": s["lambda"],
            "Status": s["status"],
            "E[ΔPN]": s["metrics"].get("expectedNetWorthChange"),
            "TC": s["metrics"].get("transactionCosts"),
            "E[ΔPN] − TC": s["metrics"].get("expectedNetWorthChange", np.nan) - s["metrics"].get("transactionCosts", np.nan),
            "σ(ΔPN)": s["metrics"].get("netWorthVolatility"),
            f"CVaR {tag} %": s["metrics"].get(f"cvar{tag}Loss"),
            "Activos USD (%)": w(s, "byCurrency", "ASSET", "USD"),
            "Equity (%)": w(s, "byType", "ASSET", "EQUITY"),
            "Pasivos USD (%)": w(s, "byCurrency", "LIABILITY", "USD"),
            "Pasivos flotantes (%)": w(s, "byRateType", "LIABILITY", "FLOAT"),
        }
        for s in an["lambdaSweep"]
    ]
).set_index("λ")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.2, 1, 1]})
ax = axes[0]
ax.plot(sweep[f"CVaR {tag} %"], sweep["E[ΔPN] − TC"], "-o", color=COLORS["ASSET"], ms=4, label="Frontera CVaR (λ)")
for lam_k, r in sweep.iterrows():
    ax.annotate(f"λ={lam_k:g}", (r[f"CVaR {tag} %"], r["E[ΔPN] − TC"]), textcoords="offset points", xytext=(4, 4), fontsize=7)
for name, alt, color, marker in [("Inicial", ini, COLORS["neutral"], "s"), (label_mv, mv, COLORS["USD"], "D"),
                                 (label_hv, hv, COLORS["LIABILITY"], "^")]:
    if alt.get("metrics"):
        m = alt["metrics"]
        ax.scatter(m[f"cvar{tag}Loss"], m["expectedNetWorthChange"] - m["transactionCosts"], color=color,
                   marker=marker, s=40, zorder=3, label=name)
ax.set(xlabel=f"CVaR {tag} % (S/ mm)", ylabel="E[ΔPN] − TC (S/ mm)", title="Frontera resultado neto vs. CVaR")
ax.legend(fontsize=7)

comp = pd.DataFrame([s.get("positions") or {} for s in an["lambdaSweep"]], index=[f"{v:g}" for v in sweep.index])
cmaps = {"PEN": "Blues", "USD": "Oranges"}
for ax, side in zip(axes[1:], ["ASSET", "LIABILITY"], strict=True):
    ids = inst.index[inst["side"] == side]
    bottom = np.zeros(len(comp))
    for ccy in sorted(inst["currency"].unique()):
        sub = [i for i in ids if inst.loc[i, "currency"] == ccy and i in comp]
        shades = colormaps[cmaps.get(ccy, "Greys")](np.linspace(0.45, 0.9, max(len(sub), 1)))
        for i, c in zip(sub, shades, strict=False):
            ax.bar(comp.index, comp[i].fillna(0.0), bottom=bottom, color=c, label=f"{i} ({ccy})", width=0.7)
            bottom += comp[i].fillna(0.0).to_numpy()
    ax.set(title=f"Composición de {SIDES[side].lower()} por λ", xlabel="λ")
    ax.legend(fontsize=6, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.15))
axes[1].set_ylabel("S/ mm")
show(fig, "barrido_lambda")
sweep.round(2)

### Incumplimientos al horizonte

Se reportan, no se imponen: el balance se revaloriza en $t_H$.

In [ ]:
breaches = pd.DataFrame(
    {
        "Prob. (%)": {k: 100 * v["probability"] for k, v in out["metrics"]["horizonBreaches"].items()},
        "Peor valor": {k: v["worst"] for k, v in out["metrics"]["horizonBreaches"].items()},
        "Causa": {k: v.get("cause") or "" for k, v in out["metrics"]["horizonBreaches"].items()},
    }
)
breaches.index = breaches.index.map(constraint_label)
print(f"Modo de vencimientos al horizonte: {horizon_mode(out['horizonMaturityLimits'])} · "
      f"costo de exigirlos en t_H: {hv.get('objectiveCost', float('nan')):.3f} S/ mm de objetivo")
breaches[breaches["Prob. (%)"] > 0].round(3)

## 8. Pruebas mínimas

Pruebas reproducibles del modelo, adaptadas de la batería del proyecto (`tests/`, pytest) para que corran aquí sin instalar nada más. Usan el mismo `input.json` y no modifican `output.json`: cada variante trabaja sobre una copia del input. Al final, una tabla resume el resultado y la celda falla si alguna prueba no pasa. Tardan menos de un minuto.

In [ ]:
import copy

TOL = 1e-6
test_results = []


def run_test(name: str, what: str, fn) -> None:
    # Ejecuta una prueba (función sin argumentos con asserts) y guarda el resultado; no detiene el notebook
    t_start = time.perf_counter()
    try:
        detail = fn() or ""
        ok = True
    except Exception as exc:  # noqa: BLE001 - se reporta cualquier falla en la tabla
        detail, ok = f"{type(exc).__name__}: {exc}", False
    test_results.append({"Prueba": name, "Qué verifica": what, "Resultado": "pasa" if ok else "FALLA",
                         "Detalle": detail, "s": round(time.perf_counter() - t_start, 1)})
    print(f"[{'pasa' if ok else 'FALLA'}] {name} · {detail}")


def with_changes(fn) -> dict:
    # Copia profunda de `doc` con una modificación; el input original no se toca
    d = copy.deepcopy(doc)
    fn(d)
    return d


def set_weight(d: dict, side: str, dimension: str, category: str, **vals) -> None:
    rows = [w for w in d["constraints"]["weights"] if (w["side"], w["dimension"], w["category"]) == (side, dimension, category)]
    assert rows, f"no existe el grupo {side}/{dimension}/{category} en input.json"
    for w in rows:
        w.update(vals)

### Caso simple resuelto a mano

Balance de juguete con solución analítica (detalle en los comentarios de la celda).

In [ ]:
# Balance de juguete: caja C1 y bono B1 (activos, B_A = 100) y un pasivo L1 fijo en 80; cinco escenarios.
# La única decisión es la caja c ∈ [10, 60]. La cola del 10 % son S4 y S5, así que, a mano:
#   E[ΔPN] = 2.88 − 0.0228 c,  CVaR_0.90 = 7.6 − 0.082 c,  TC = 0.002 |c − 30|.
# Con λ = 0.5 la pendiente para c > 30 es −0.0248 + 0.041 > 0  ⇒  c* = 60 (tope de caja).
#   E = 1.512, CVaR = 2.68, TC = 0.06, objetivo = 1.512 − 0.5·2.68 − 0.06 = 0.112.
TOY_SCEN = ["S1", "S2", "S3", "S4", "S5"]
TOY_PROBS = np.array([0.40, 0.30, 0.20, 0.06, 0.04])
TOY_R_BOND = np.array([0.08, 0.06, 0.04, -0.02, -0.10])
TOY_BOND_FLOWS = np.array([0.05, 0.05, 0.05, 0.05, 0.0])


def toy_doc(min_cash: float = 8.0) -> dict:
    def inst_(iid, side, typ, mv, maturity):
        return {"id": iid, "side": side, "name": iid, "instrument_type": typ, "currency": "PEN",
                "market_value_pen": mv, "notional_native": mv, "fixed_coupon": None, "frequency": None,
                "maturity": maturity, "reference_factor": None, "spread": None, "reset_months": None,
                "other_market_ref": None, "notes": ""}

    weights = [("ASSET", "TYPE", "CASH", 0.10, 0.60), ("ASSET", "TYPE", "FIXED_INCOME", 0.40, 0.90),
               ("ASSET", "CURRENCY", "PEN", 0.0, 1.0), ("LIABILITY", "CURRENCY", "PEN", 0.0, 1.0),
               ("LIABILITY", "RATE_TYPE", "FIXED", 0.0, 1.0)]
    return {
        "schemaVersion": "1.0", "caseId": "TOY", "valuationDate": "2025-12-31", "horizonDate": "2026-12-31",
        "baseCurrency": "PEN",
        "instruments": [inst_("C1", "ASSET", "CASH", 30.0, None), inst_("B1", "ASSET", "BOND_FIXED", 70.0, "2030-12-31"),
                        inst_("L1", "LIABILITY", "BOND_FIXED", 80.0, "2030-12-31")],
        "constraints": {
            "weights": [{"index": k, "side": s, "dimension": dm, "category": c, "min_weight": lo, "max_weight": hi,
                         "note": ""} for k, (s, dm, c, lo, hi) in enumerate(weights)],
            "common": {"minNetWorth": 15.0, "maxLiabilitiesToAssets": 0.85, "minCash": min_cash, "assetBudget": 100.0,
                       "liabilityBudget": 80.0, "allowShort": False},
        },
        "transactionCosts": {
            "C1": {"side": "ASSET", "restructuring_cost": 0.0, "reduction_prepayment_cost": 0.0},
            "B1": {"side": "ASSET", "restructuring_cost": 0.002, "reduction_prepayment_cost": 0.0},
            "L1": {"side": "LIABILITY", "restructuring_cost": 0.01, "reduction_prepayment_cost": 0.01},
        },
        "fundingAlternatives": [], "stressScenarios": [],
        "caseParameters": {"horizonDate": "2026-12-31", "cvarAlpha": 0.90, "riskAversionLambda": 0.5,
                           "lambdaGrid": [0.0, 0.5], "riskMeasure": "CVAR", "bootstrapCount": 4,
                           "bootstrapProbability": 0.96, "stressProbability": 0.04, "seed": 1},
    }


def toy_components():
    # Rendimientos por S/ (reval + flujos): caja 3 %, bono TOY_R_BOND, pasivo 3 % en todos los escenarios
    ids = ["C1", "B1", "L1"]
    reval = pd.DataFrame([np.full(5, 0.03), TOY_R_BOND - TOY_BOND_FLOWS, np.full(5, 0.03 - 0.04)], index=ids,
                         columns=TOY_SCEN)
    flows = pd.DataFrame([np.zeros(5), TOY_BOND_FLOWS, np.full(5, 0.04)], index=ids, columns=TOY_SCEN)
    scen_ = risk.ScenarioSet(ids=list(TOY_SCEN), deltas=pd.DataFrame(0.0, index=TOY_SCEN, columns=risk.FACTORS),
                             probs=TOY_PROBS.copy(), kind=["bootstrap"] * 4 + ["stress"])
    return reval, flows, scen_


def test_toy_by_hand():
    res = opt.solve_data(opt.prepare(toy_doc(), components=toy_components()))
    assert res["status"] == "OPTIMAL", res["status"]
    got = {k: float(res["positions"][k]) for k in ["C1", "B1", "L1"]}
    for k, v in {"C1": 60.0, "B1": 40.0, "L1": 80.0}.items():
        assert abs(got[k] - v) < TOL, (k, got[k])
    assert abs(res["objective"] - 0.112) < TOL, res["objective"]
    assert abs(res["cvar_lp"] - 2.68) < TOL, res["cvar_lp"]
    assert abs(res["transaction_cost"] - 0.06) < TOL, res["transaction_cost"]
    return f"caja {got['C1']:.4f}, bono {got['B1']:.4f}, objetivo {res['objective']:.6f} (a mano 0.112)"


run_test("1. Caso simple resuelto a mano", "Juguete de 2 activos, 1 pasivo y 5 escenarios: posiciones, CVaR, TC "
         "y objetivo iguales a la solución analítica (tol. 1e-6)", test_toy_by_hand)

### Pruebas sobre el input real

CVaR del LP, presupuestos y límites, shock extremo de tasas, restricciones infactibles, parámetros leídos del input, monotonía en λ y calibración.

In [ ]:
def test_lp_cvar_ex_post():
    # El CVaR del LP (ζ + Σ p u / (1 − α)) coincide con el CVaR recalculado ex post, en el óptimo y en el barrido
    ev_opt = opt.evaluate(data, pos["optimal"])
    gaps = [abs(ev_opt["cvar"] - out["metrics"][f"cvar{tag}Loss"])]
    for lam_k in doc["caseParameters"]["lambdaGrid"]:
        if lam_k > 0:
            res = opt.solve_data(data, lam=lam_k)
            gaps.append(abs(res["cvar_lp"] - opt.evaluate(data, res["positions"])["cvar"]))
    assert max(gaps) < 1e-5, gaps
    return f"máx. diferencia {max(gaps):.1e} S/ mm en {len(gaps)} soluciones"


def test_budgets_and_limits():
    # Presupuestos, no negatividad y todos los límites del input en la solución óptima
    x = pos["optimal"]
    side = inst["side"].reindex(x.index)
    gap_a = abs(x[side == "ASSET"].sum() - com["assetBudget"])
    gap_l = abs(x[side == "LIABILITY"].sum() - com["liabilityBudget"])
    assert gap_a < TOL and gap_l < TOL, (gap_a, gap_l)
    assert (x >= -TOL).all(), x[x < -TOL]
    bad = [c["name"] for c in out["constraintChecks"] if not c["ok"]]
    assert not bad, bad
    return f"Σ activos = {com['assetBudget']:g}, Σ pasivos = {com['liabilityBudget']:g}, mín. posición {x.min():.2e}, " \
           f"{len(out['constraintChecks'])} límites ok"


def test_short_rate_shock():
    # +500 pb en los nodos cortos (stressTenors.short) de PEN y USD en todos los escenarios: la exposición neta a la
    # tasa corta (caja + activos indexados al tramo corto − pasivos indexados a él) debe subir
    short_nodes = doc["caseParameters"]["stressTenors"]["short"]["nodes"]
    short_factors = [f for f in risk.FACTORS if "_SPOT_" in f and f.split("_SPOT_")[1] in short_nodes]
    segs = {s["id"] for s in doc["curveSegments"] if s["fromNode"] in short_nodes and s["toNode"] in short_nodes}
    ref_seg = inst["reference_factor"].map(lambda r: r.split("_")[-1] if isinstance(r, str) else None)
    short = (inst["instrument_type"] == "CASH") | ref_seg.isin(segs)
    deltas = scen.deltas.copy()
    deltas[short_factors] += 0.05
    reval_s, flows_s = risk.scenario_components(doc, deltas)
    scen_s = risk.ScenarioSet(scen.ids, deltas, scen.probs, scen.kind)
    shocked = opt.solve_data(opt.prepare(doc, components=(reval_s, flows_s, scen_s)))
    assert shocked["status"] == "OPTIMAL", shocked["status"]

    def agg(x, s):
        return float(x.reindex(inst.index[short & (inst["side"] == s)]).sum())

    a0, l0 = agg(pos["optimal"], "ASSET"), agg(pos["optimal"], "LIABILITY")
    a1, l1 = agg(shocked["positions"], "ASSET"), agg(shocked["positions"], "LIABILITY")
    assert a1 >= a0 - TOL and l1 <= l0 + TOL, (a0, a1, l0, l1)
    assert (a1 - l1) >= (a0 - l0) + 1.0, (a0 - l0, a1 - l1)
    return f"activos cortos {a0:.1f} → {a1:.1f}; pasivos cortos {l0:.1f} → {l1:.1f} S/ mm"


def test_infeasible():
    # Límites contradictorios: el programa informa INFEASIBLE, sin excepción y con un output que cumple el contrato
    cash_max = next(w["max_weight"] for w in doc["constraints"]["weights"]
                    if (w["side"], w["dimension"], w["category"]) == ("ASSET", "TYPE", "CASH"))
    cases = {
        "caja mínima > caja máxima": lambda d: d["constraints"]["common"].update(
            minCash=1.25 * cash_max * com["assetBudget"]),
        "mínimos de moneda suman 110 %": lambda d: (set_weight(d, "ASSET", "CURRENCY", "PEN", min_weight=0.90),
                                                    set_weight(d, "ASSET", "CURRENCY", "USD", min_weight=0.20)),
    }
    for label, mutate in cases.items():
        bad = with_changes(mutate)
        res = opt.solve(bad, components=comps)
        assert res["status"] == "INFEASIBLE", (label, res["status"])
        assert validate_output(res) == [], (label, validate_output(res))
    toy = opt.solve_data(opt.prepare(toy_doc(min_cash=70.0), components=toy_components()))
    assert toy["status"] == "INFEASIBLE" and toy["positions"] is None, toy["status"]
    return f"{len(cases) + 1} casos → INFEASIBLE con output válido"


def test_no_hardcoded_parameters():
    # Cambiar datos de input.json cambia la solución: concentración máx. de activos 30 % y λ = 5
    b_a = com["assetBudget"]
    conc = with_changes(lambda d: set_weight(d, "ASSET", "CONCENTRATION", "SINGLE_POSITION", max_weight=0.30))
    r1 = opt.solve_data(opt.prepare(conc, components=comps))
    assets = inst.index[inst["side"] == "ASSET"]
    assert r1["positions"][assets].max() <= 0.30 * b_a + TOL, r1["positions"][assets].max()
    d1 = (r1["positions"] - pos["optimal"]).abs().max()
    r2 = opt.solve_data(opt.prepare(with_changes(lambda d: d["caseParameters"].update(riskAversionLambda=5.0)),
                                    components=comps))
    d2 = (r2["positions"] - pos["optimal"]).abs().max()
    assert d1 > 1.0 and d2 > 1.0, (d1, d2)
    return f"concentración 30 %: máx. activo {r1['positions'][assets].max():.1f} (≤ {0.30 * b_a:g}); " \
           f"cambio máx. {d1:.1f} y {d2:.1f} (λ = 5) S/ mm"


def test_lambda_monotonicity():
    # Más aversión al riesgo ⇒ CVaR y E[ΔPN] − TC no crecen (preferencia revelada)
    sw = [s for s in an["lambdaSweep"] if s["status"] in opt.OPTIMAL]
    cvar = [s["metrics"][f"cvar{tag}Loss"] for s in sw]
    net = [s["metrics"]["expectedNetWorthChange"] - s["metrics"]["transactionCosts"] for s in sw]
    assert len(sw) == len(an["lambdaSweep"]), "hay λ sin solución óptima"
    assert all(b <= a + 1e-5 for a, b in zip(cvar, cvar[1:], strict=False)), cvar
    assert all(b <= a + 1e-5 for a, b in zip(net, net[1:], strict=False)), net
    return f"CVaR {cvar[0]:.1f} → {cvar[-1]:.1f}; E − TC {net[0]:.2f} → {net[-1]:.2f} ({len(sw)} valores de λ)"


def test_calibration():
    # Los spreads calibrados reproducen el valor de mercado en t0
    err = float(cal["Error (S/ mm)"].abs().max())
    assert err < TOL, err
    return f"máx. |V0 − mercado| = {err:.1e} S/ mm"


run_test("2. CVaR del LP = CVaR ex post", "ζ + Σ p u/(1 − α) del LP coincide con el CVaR recalculado, en el óptimo "
         "y en cada λ del barrido", test_lp_cvar_ex_post)
run_test("3. Presupuestos y límites", "Σ activos = B_A, Σ pasivos = B_L, sin cortos y todos los límites cumplidos",
         test_budgets_and_limits)
run_test("4. Shock extremo de tasas (+500 pb)", "Con +500 pb en los nodos cortos de ambas curvas sube la exposición "
         "neta a la tasa corta", test_short_rate_shock)
run_test("5. Restricciones infactibles", "Límites contradictorios ⇒ INFEASIBLE sin excepción y output válido",
         test_infeasible)
run_test("6. Sin parámetros en el código", "Cambiar la concentración máxima o λ en el input cambia la solución",
         test_no_hardcoded_parameters)
run_test("7. Monotonía en λ", "Al subir λ, CVaR y E[ΔPN] − TC no crecen", test_lambda_monotonicity)
run_test("8. Calibración de spreads", "V0 del modelo = valor de mercado en t0 (tol. 1e-6)", test_calibration)

### Resumen

In [ ]:
tests_tab = pd.DataFrame(test_results).set_index("Prueba")
display(tests_tab)
n_fail = int((tests_tab["Resultado"] != "pasa").sum())
print(f"{len(tests_tab) - n_fail} de {len(tests_tab)} pruebas pasan")
assert n_fail == 0, f"{n_fail} pruebas fallan: revisar la tabla"

## 9. Sensibilidades (opcional)

Se corren solo si `RUN_SENSITIVITY = True`.

In [ ]:
if RUN_SENSITIVITY:
    sens = sensitivity.run_all(doc)
    Path(OUTPUT_PATH).with_name("sensitivity.json").write_text(
        json.dumps(_clean(sens), indent=2, ensure_ascii=False), encoding="utf-8")
    if "error" in sens:
        raise RuntimeError(sens["error"])
    print(f"Sensibilidades en {sens['meta']['runtimeSeconds']:.0f} s → sensitivity.json")
else:
    sens = None
    print("RUN_SENSITIVITY = False: se omiten S1–S9")

In [ ]:
if sens:
    cfg = cp_["sensitivities"]
    base_rule = cp_.get("floatingCouponRule") or "FLAT"
    rs = pd.DataFrame(sens["rateShocks"])
    names = {"initial": "Inicial", "optimal": label_opt, "meanVariance": "Mínima varianza"}
    order = [s["id"] for s in cfg["rateShocks"]]
    s1 = (rs[rs["rule"] == base_rule].pivot(index="shock", columns="portfolio", values="netWorth")
          .reindex(index=order, columns=[c for c in names if c in set(rs["portfolio"])]).rename(columns=names))
    for rule in sorted(set(rs["rule"]) - {base_rule}):
        sub = rs[(rs["rule"] == rule) & (rs["portfolio"] == "optimal")].set_index("shock")["netWorth"]
        s1[f"Óptimo, cupones {coupon_rule(rule)}"] = sub.reindex(order)
    fig, ax = plt.subplots(figsize=(11, 4))
    k = np.arange(len(s1))
    colors = [COLORS["neutral"], COLORS["ASSET"], COLORS["USD"], COLORS["PEN"], COLORS["EQUITY"]]
    width = 0.8 / len(s1.columns)
    for j, (c, color) in enumerate(zip(s1.columns, colors, strict=False)):
        ax.bar(k + (j - (len(s1.columns) - 1) / 2) * width, s1[c], width, color=color, label=c,
               alpha=0.5 if c == "Inicial" else 1.0)
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xticks(k, s1.index, fontsize=7)
    ax.set(ylabel="ΔPN (S/ mm)", title="S1 · ΔPN instantáneo ante shocks de tasas (revalorización completa)")
    ax.legend(fontsize=7, ncol=4)
    show(fig, "sensibilidad_tasas")
    display(s1.round(2))

In [ ]:
if sens:
    sp = pd.DataFrame(sens["shadowPrices"])
    if len(sp):
        sp["Límite"] = [" / ".join(constraint_label(n) for n in name.split("+")) for name in sp["name"]]
        s2 = sp.set_index("Límite")[["bound", "limit", "dualPerPp", "finiteDiffSmallPerPp", "finiteDiffPerPp"]]
        s2 = s2.sort_values("dualPerPp")
        fig, ax = plt.subplots(figsize=(9, 0.35 * len(s2) + 1))
        ax.barh(np.arange(len(s2)), s2["dualPerPp"], color=COLORS["ASSET"])
        ax.set_yticks(np.arange(len(s2)), s2.index, fontsize=7)
        ax.set(xlabel="S/ mm de objetivo por 1 pp de relajación", title="S2 · Precios sombra de los límites activos")
        show(fig, "precios_sombra")
        display(s2.round(4))

In [ ]:
if sens:
    def net(m: dict) -> float:
        return m["expectedNetWorthChange"] - m["transactionCosts"]

    def summary(group: str, label: str, row: dict) -> dict:
        ev_ = row.get("basePortfolio") or row.get("outOfSample")
        ok = row["status"] in opt.OPTIMAL
        return {"Grupo": group, "Variante": label, "Status": row["status"], "E − TC": net(row) if ok else None,
                "CVaR": row["cvarLoss"], "Distancia L1 a x*": row["distanceToBase"],
                "E − TC de x*": net(ev_) if ev_ else None, "CVaR de x*": ev_["cvarLoss"] if ev_ else None}

    base = sens["base"]
    rows = [summary("Base", "Base", {**base, "distanceToBase": 0.0, "basePortfolio": base})]
    rows += [summary("S3 costos", f"TC × {r['multiplier']:g}", r) for r in sens["costs"]]
    rows += [summary("S4 α", f"α = {r['alpha']:g}", r) for r in sens["alphas"]]
    rows += [summary("S5 cola", f"P estrés = {r['stressProbability']:g}", r) for r in sens["stressWeight"]]
    rows += [summary("S5 cola", f"sin {r['dropped']}", r) for r in sens["stressLeaveOneOut"]]
    rows += [summary("S6 semillas", f"semilla {r['seed']}", r) for r in sens["seeds"]]
    rows += [summary("S7 deriva", f"deriva {r['variant']}", r) for r in sens["drift"]]
    rows += [summary("S8 estrés", f"+ {r['scenario']}", r) for r in sens["extraStress"]]
    rows += [summary("S9 cupones", f"cupones {coupon_rule(r['rule'])}", r) for r in sens["floatingCouponRule"]]
    s39 = pd.DataFrame(rows).set_index("Variante")

    fig_rows = s39[~s39["Grupo"].isin(["S3 costos", "S4 α"])].iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(fig_rows) + 1.2))
    k = np.arange(len(fig_rows))
    ax.barh(k + 0.2, fig_rows["CVaR de x*"], 0.4, color=COLORS["neutral"], alpha=0.6, label="x* sin reoptimizar")
    ax.barh(k - 0.2, fig_rows["CVaR"], 0.4, color=COLORS["ASSET"], label="Reoptimizada")
    ax.axvline(base["cvarLoss"], color="black", ls=":", lw=1, label=f"CVaR base ({base['cvarLoss']:.1f})")
    ax.set_yticks(k, fig_rows.index, fontsize=7)
    ax.set(xlabel=f"CVaR {tag} % (S/ mm)", title="S3–S9 · Robustez del CVaR ante cambios de modelo")
    ax.legend(fontsize=7, loc="lower right")
    show(fig, "robustez_cvar")
    display(s39.round(2))

## Resultado

`output.json` (ruta en `OUTPUT_PATH`) sigue el contrato v1.0 del Anexo 1.B:

- `status`, `objectiveValue`: estado del solver y valor de $E[\Delta PN] - \lambda\,\mathrm{CVaR} - TC$.
- `positions`: montos inicial, óptimo y cambio por instrumento (S/ mm).
- `metrics`: $E[\Delta PN]$, TC, σ, VaR/CVaR, pérdidas por estrés e incumplimientos al horizonte.
- `weights`, `constraintChecks`: composición por moneda, tipo, tasa y vencimiento y holgura de cada límite.
- `riskModel`: escenarios, semilla, deriva y covarianza usados.
- `analysis`: posición inicial, barrido de λ, benchmark media-varianza y variante con vencimientos exigidos al
  horizonte.

Si se activó `RUN_SENSITIVITY`, junto a él queda `sensitivity.json` con S1–S9.